# Polyamine SMS cMOS iPSC Analysis

## Purpose

This notebook contains the analysis workflow used to process and quantify high-density MaxOne cMOS recordings from human iPSC-derived neuronal networks in the AAV-Ctrl and AAV-SMS groups during baseline, carbamazepine (CBZ) exposure, and washout conditions. It is organized as a reproducible end-to-end pipeline that converts exported unit-position and time-series files into neuronal, network-burst, frequency-domain, and functional-connectivity measurements used for statistical comparison and figure generation.

## Analysis workflow

The notebook performs the following major steps:

1. **Data consolidation and metadata assignment**  
   Unit-position and neuronal time-series CSV files are combined into multi-sheet Excel workbooks. Sample identifiers are assigned to AAV-Ctrl or AAV-SMS groups, and recording conditions are mapped to Baseline, CBZ30uM, CBZ60uM, CBZ100uM, or Washout.

2. **Per-neuron data assembly**  
   Unit positions, burst measurements, and neuronal time series are matched by unit identifier. The resulting tables preserve sample, group, condition, spatial coordinates, and neuronal activity data in a common structure.

3. **Rate-histogram and neuronal activity analysis**  
   Neuronal spike trains are converted to firing-rate histograms and smoothed for downstream network analysis. Spatial activity maps are generated to visualize the location and activity of recorded neuronal units across experimental conditions.

4. **Burst analysis**  
   Burst parameters are summarized at neuronal and MEA levels. The workflow generates raw and baseline-normalized comparisons, statistical tables, and publication figures. Selected-condition analyses used for the manuscript exclude CBZ60uM where indicated.

5. **Network-burst analysis**  
   Population firing-rate histograms are used to detect network-burst peaks and quantify network-burst amplitude, frequency, width, synchronization-related measures, and associated summary statistics. Raster plots, rate histograms, and network-burst visualizations are exported for inspection.

6. **Frequency-domain analysis**  
   Frequency-domain percentages are calculated from network activity, compared across experimental groups and CBZ conditions, normalized to the corresponding baseline values, and summarized across defined frequency intervals. Source data used for the plotted means, variability, and individual observations are exported to Excel where implemented.

7. **Functional-connectivity analysis**  
   Pairwise cross-correlation and Granger-causality analyses are used to derive functional connectivity matrices. Network-level connectivity parameters are then calculated and compared between groups and conditions, including normalized analyses.

8. **Connectivity visualization**  
   Connectivity matrices are combined with neuronal spatial coordinates to generate geometric network maps and condition-wise visualizations.

## Reproducible file organization

All local machine-specific paths have been removed. The notebook uses paths relative to the repository root:

- `data/` — input data required to run the analysis.
- `results/` — intermediate files, statistical tables, figures, and connectivity outputs generated by the notebook.

The `results/` directory is created automatically. Input files should be placed in the locations defined in the configuration cell below.

## Notes

The notebook preserves the analysis logic of the original workflow while removing machine-specific directory references and line-by-line development comments. The code is intended to document the analysis used for the associated manuscript and to make the computational workflow easier to inspect, reproduce, and adapt.


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"

POSITIONS_DIR = DATA_DIR / "01_Positions_CBZ01"
TIMESERIES_DIR = DATA_DIR / "01_Timeseries_CBZ01"

UNIT_POSITION_XLSX = RESULTS_DIR / "02_UnitPosition_SMS_CBZ_All_01.xlsx"
TIMESERIES_XLSX = RESULTS_DIR / "02_Timeseries_SMS_CBZ_All_01.xlsx"
BURST_XLSX = DATA_DIR / "02_Burst_SMS_CBZ_01.xlsx"

UNIT_BURST_CSV = RESULTS_DIR / "03_UnitPos_Burst_CBZ_01.csv"
NORMALIZED_UNIT_BURST_CSV = RESULTS_DIR / "03_UnitPos_NormBurst_CBZ_01.csv"

UNIT_TIMESERIES_PKL = RESULTS_DIR / "03_UnitPos_TimSeries_CBZ_01.pkl.gz"
RATE_HISTOGRAM_PKL = RESULTS_DIR / "03_UnitPos_TimSerRateHist_CBZ_01.pkl.gz"

NETWORK_BURST_XLSX = RESULTS_DIR / "05_01_NetworkBurstRH_MEABasedStat_01.xlsx"
NORMALIZED_FREQUENCY_XLSX = RESULTS_DIR / "05_07_NormRHperBin_01.xlsx"

CONNECTIVITY_MATRIX_DIR = RESULTS_DIR / "06_GCCC_CBZ_Matrices_01"
CONNECTIVITY_PARAMETERS_CSV = RESULTS_DIR / "07_00_Net_Params_Combined.csv"

BASELINE_CONNECTIVITY_PKL = DATA_DIR / "01_UnitPos_TimSeries_merged_01.pkl"
BASELINE_CCGC_PKL = DATA_DIR / "01_UnitPos_TimSeries_merged_ex.pkl.gz"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)


## 02 Combine all neuronal unit Position files in one xlsx file 


In [ ]:
import os
import pandas as pd

def csv_to_excel(csv_folder, output_excel):

    with pd.ExcelWriter(output_excel, engine='xlsxwriter') as writer:

        for csv_file in os.listdir(csv_folder):

            if csv_file.endswith('.csv'):

                csv_path = os.path.join(csv_folder, csv_file)
                df = pd.read_csv(csv_path)

                sheet_name = csv_file[20:-4]

                df.to_excel(writer, sheet_name=sheet_name, index=False)

csv_folder = POSITIONS_DIR
output_excel = UNIT_POSITION_XLSX
csv_to_excel(csv_folder, output_excel)


## 02 Combine all timeseries data in one xlsx file (apply sampling frequency)


In [ ]:
import os
import pandas as pd

def csv_to_excel(csv_folder, output_excel, sampling_frequency):

    with pd.ExcelWriter(output_excel, engine='xlsxwriter') as writer:

        for csv_file in os.listdir(csv_folder):

            if csv_file.endswith('.csv'):

                csv_path = os.path.join(csv_folder, csv_file)
                df = pd.read_csv(csv_path)

                sheet_name = csv_file[22:-4]

                df.iloc[:, 1:] = df.iloc[:, 1:].apply(lambda x: x / sampling_frequency)

                df.to_excel(writer, sheet_name=sheet_name, index=False)

csv_folder = TIMESERIES_DIR
output_excel = TIMESERIES_XLSX
sampling_frequency = 20000

csv_to_excel(csv_folder, output_excel, sampling_frequency)


## 03 Merging UnitPos data with Burst Data
###### Burst data has already beene xtracted using Neuroexplorer software


In [ ]:
import pandas as pd
import os

unit_position_file = UNIT_POSITION_XLSX
burst_file = BURST_XLSX
ctrl_samples = ['18389', '18505', '20810']
sms_samples = ['18406', '18412', '18581']

def identify_group(sample_id):
    if sample_id in ctrl_samples:
        return 'AAV-Ctrl'
    elif sample_id in sms_samples:
        return 'AAV-SMS'
    else:
        return 'Unknown'

condition_map = {
    '01': 'Baseline',
    '02': 'CBZ30uM',
    '03': 'CBZ60uM',
    '04': 'CBZ100uM',
    '05': 'Washout'
}

unit_position_sheets = pd.ExcelFile(unit_position_file).sheet_names
burst_sheets = pd.ExcelFile(burst_file).sheet_names

merged_data = []

for sheet_name in unit_position_sheets:

    date_info, sample_id, condition_suffix = sheet_name.split('_')

    condition = condition_map.get(condition_suffix, 'Unknown')

    unit_position_df = pd.read_excel(unit_position_file, sheet_name=sheet_name)
    burst_df = pd.read_excel(burst_file, sheet_name=sheet_name)
    burst_df = burst_df[burst_df['Variable'] != 'nr_001']

    merged_df = pd.merge(unit_position_df, burst_df,
                         how='inner',
                         left_on='unit_id',
                         right_on='Variable')

    merged_df['Date'] = date_info
    merged_df['Sample_ID'] = sample_id
    merged_df['Group'] = identify_group(sample_id)

    merged_df.insert(4, 'Condition', condition)

    merged_data.append(merged_df)

final_df = pd.concat(merged_data, ignore_index=True)

output_file = UNIT_BURST_CSV
final_df.to_csv(output_file, index=False)
print(f"Merged data has been saved as {output_file}")


## 03 Merge UnitPos Condition and Timeseries for ---->- Pickle file


In [ ]:
import pandas as pd
import numpy as np
import os
import pickle
import zipfile

position_file = UNIT_POSITION_XLSX
timeseries_file = TIMESERIES_XLSX
position_sheets = pd.read_excel(position_file, sheet_name=None)
timeseries_sheets = pd.read_excel(timeseries_file, sheet_name=None)

ctrl_samples = ['18389', '18505', '20810']
sms_samples = ['18406', '18412', '18581']

def identify_group(sample_id):
    if sample_id in ctrl_samples:
        return 'AAV-Ctrl'
    elif sample_id in sms_samples:
        return 'AAV-SMS'
    else:
        return 'Unknown'

date_to_dpi_postSMS = {
    '20240722': (135, 0),
    '20240730': (135, 0),
    '20240731': (136, 1),
    '20240801': (137, 2),
    '20240805': (141, 6),
    '20240808': (144, 9),
    '20240812': (148, 13),
    '20240814': (150, 15),
    '20240821': (157, 22),
    '20240910': (177, 42),
    '20241008': (205, 70)
}

condition_map = {
    '01': 'Baseline',
    '02': 'CBZ30uM',
    '03': 'CBZ60uM',
    '04': 'CBZ100uM',
    '05': 'Washout'
}

output_data = []

for sheet_name, position_data in position_sheets.items():
    if sheet_name in timeseries_sheets:
        print(f"Processing: {sheet_name}")
        timeseries_data = timeseries_sheets[sheet_name]

        timeseries_data = timeseries_data.iloc[:, 1:]
        date, sample_id, condition_suffix = sheet_name.split('_')
        group = identify_group(sample_id)
        dpi, post_SMS = date_to_dpi_postSMS.get(date, (None, None))
        condition = condition_map.get(condition_suffix, 'Unknown')
        timeseries_data.columns = timeseries_data.columns.astype(int)
        ordered_neurons = position_data['unit_id'].tolist()
        matching_neurons = [neuron for neuron in ordered_neurons if neuron in timeseries_data.columns]

        print(f"Matching neurons for {sheet_name} (after reordering based on position data):")
        print(matching_neurons)
        if len(matching_neurons) == 0:
            print(f"No matching neurons found for {sheet_name}.")
            continue

        ordered_timeseries_data = timeseries_data[matching_neurons]
        transposed_timeseries_data = ordered_timeseries_data.transpose()

        for neuron_id in matching_neurons:
            neuron_timeseries = transposed_timeseries_data.loc[neuron_id].dropna().tolist()
            neuron_timeseries = neuron_timeseries if neuron_timeseries else []
            neuron_x = position_data.loc[position_data['unit_id'] == neuron_id, 'x'].values[0]
            neuron_y = position_data.loc[position_data['unit_id'] == neuron_id, 'y'].values[0]

            output_data.append({
                'Group': group,
                'File Name': sheet_name,
                'Sample_ID': sample_id,
                'Date': date,
                'dpi': dpi,
                'post_SMS': post_SMS,
                'unit_id': neuron_id,
                'x': neuron_x,
                'y': neuron_y,
                'Condition': condition,
                'Timeseries': neuron_timeseries
            })

if len(output_data) > 0:
    print("Sample data collected:", output_data[:5])
else:
    print("No data was collected. Check the input files and sheet names.")

output_df = pd.DataFrame(output_data)
print("Available columns:", output_df.columns)

if 'Date' in output_df.columns:
    output_df['Date'] = output_df['Date'].astype(str)
    output_df['unit_id'] = output_df['unit_id'].astype(str)
else:
    print("Error: 'Date' column is missing from the DataFrame.")

output_path = RESULTS_DIR
output_compressed_pickle_path = UNIT_TIMESERIES_PKL
output_df.to_pickle(output_compressed_pickle_path, compression='gzip')

print(f"Data has been exported successfully to {output_compressed_pickle_path}")


## 03_ Calculate & Smooth RateHisto (RH): Save RH and Ts --> zippeded pickle format 


In [ ]:
import pandas as pd
import numpy as np
import os
from scipy.ndimage import gaussian_filter1d

input_pickle_path = UNIT_TIMESERIES_PKL
output_df = pd.read_pickle(input_pickle_path)

bin_start = 0
bin_end = 600
bin_size = 0.1
bins = np.arange(bin_start, bin_end + bin_size, bin_size)
left_bins = bins[:-1]

def calculate_tmean_freq(timeseries_data, bins, bin_size):
    spike_counts, _ = np.histogram(timeseries_data, bins=bins)
    tmean_freq = spike_counts / bin_size
    return tmean_freq

def smooth_rate_histogram(rate_histogram, sigma=1.5):
    return gaussian_filter1d(rate_histogram, sigma=sigma)

output_df['Rate_Histo'] = output_df['Timeseries'].apply(lambda ts: calculate_tmean_freq(ts, bins, bin_size))
output_df['Rate_Histo'] = output_df['Rate_Histo'].apply(lambda rh: smooth_rate_histogram(rh).tolist())
output_df['Left_Bins'] = [left_bins.tolist()] * len(output_df)

cols = list(output_df.columns)
timeseries_index = cols.index('Timeseries')
cols.insert(timeseries_index + 1, cols.pop(cols.index('Rate_Histo')))
cols.insert(timeseries_index + 2, cols.pop(cols.index('Left_Bins')))
output_df = output_df[cols]

output_compressed_pickle_path = RATE_HISTOGRAM_PKL

output_df.to_pickle(output_compressed_pickle_path, compression='gzip')

print(f"Data has been exported successfully to {output_compressed_pickle_path}")


## Read and check the Pickle file


In [ ]:
import pandas as pd

output_pickle_path = RATE_HISTOGRAM_PKL
loaded_df = pd.read_pickle(output_pickle_path)

print(f"Type of the loaded object: {type(loaded_df)}")
print(f"DataFrame shape: {loaded_df.shape}")
print("Data types of each column:")
print(loaded_df.dtypes)
print("DataFrame structure overview:")
loaded_df.info()
print("First few rows of the DataFrame:")
print(loaded_df.head(50))

neuron_data = loaded_df.iloc[3]
left_bins = neuron_data['Left_Bins']
rate_histo = neuron_data['Rate_Histo']
print(f"\nNeuron: {neuron_data['unit_id']}")
print(f"Length of Left_Bins: {len(left_bins)}")
print(f"Length of Rate_Histo: {len(rate_histo)}")

print(f"\nLeft_Bins (first 10 values): {left_bins[:10]}")
print(f"Rate_Histo (first 10 values): {rate_histo[:10]}")


## 03Plots_ Plot Neuron Maps based on Condition 


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import cv2
import numpy as np

csv_file = UNIT_BURST_CSV
final_df = pd.read_csv(csv_file)
output_dir = RESULTS_DIR / "03Plots_01_NeuronsMappedPosCondition"
os.makedirs(output_dir, exist_ok=True)

video_frames = {}

condition_order = ['Baseline', 'CBZ30uM', 'CBZ60uM', 'CBZ100uM', 'Washout']

for (sample_id, condition) in final_df.groupby(['Sample_ID', 'Condition']).groups.keys():
    sample_df = final_df[(final_df['Sample_ID'] == sample_id) & (final_df['Condition'] == condition)]

    x_positions = sample_df['x']
    y_positions = sample_df['y']

    plt.figure(figsize=(12, 6), facecolor='black')
    ax = plt.gca()
    ax.set_facecolor('black')

    plt.scatter(x_positions, y_positions, edgecolor='gray', facecolor=(1, 0.5, 0, 0.5),
                s=20, alpha=0.7, linewidth=0.5, label=f'Sample_ID: {sample_id}, Condition: {condition}')
    plt.title(f'Neuron Positions for Sample_ID: {sample_id} (Condition={condition})', color='white')
    plt.xlabel('X Position', color='white')
    plt.ylabel('Y Position', color='white')

    plt.xlim(0, 4000)
    plt.ylim(0, 2200)
    ax.tick_params(colors='white')
    rect = plt.Rectangle((0, 0), 4000, 2200, linewidth=2, edgecolor='white', facecolor='none')
    ax.add_patch(rect)

    output_filename = f'{sample_id}_condition_{condition}.pdf'
    output_path = os.path.join(output_dir, output_filename)
    plt.savefig(output_path, format='pdf', bbox_inches='tight', dpi=300)

    image_output_path = os.path.join(output_dir, f'{sample_id}_condition_{condition}.png')
    plt.savefig(image_output_path, format='png', bbox_inches='tight', dpi=300)

    if sample_id not in video_frames:
        video_frames[sample_id] = {}
    video_frames[sample_id][condition] = image_output_path

    plt.close()

def resize_image(image, target_size):
    return cv2.resize(image, target_size, interpolation=cv2.INTER_LANCZOS4)

def interpolate_images(img1, img2, steps=10):
    interpolated_images = []
    for i in range(steps):
        alpha = i / float(steps)
        beta = 1.0 - alpha
        blended_img = cv2.addWeighted(img1, beta, img2, alpha, 0)
        interpolated_images.append(blended_img)
    return interpolated_images

for sample_id, condition_images in video_frames.items():

    sorted_image_paths = [condition_images[condition] for condition in condition_order if condition in condition_images]

    if len(sorted_image_paths) == len(condition_images):
        video_output_path = os.path.join(output_dir, f'{sample_id}_plots_video.mp4')
        first_image = cv2.imread(sorted_image_paths[0])
        height, width, _ = first_image.shape
        target_size = (width, height)
        video_writer = cv2.VideoWriter(video_output_path, cv2.VideoWriter_fourcc(*'mp4v'), 10, (width, height))

        for i in range(len(sorted_image_paths) - 1):
            img1 = cv2.imread(sorted_image_paths[i])
            img2 = cv2.imread(sorted_image_paths[i + 1])

            img1_resized = resize_image(img1, target_size)
            img2_resized = resize_image(img2, target_size)

            video_writer.write(img1_resized)

            interpolated_frames = interpolate_images(img1_resized, img2_resized, steps=10)
            for interpolated_frame in interpolated_frames:
                video_writer.write(interpolated_frame)

        final_img = cv2.imread(sorted_image_paths[-1])
        final_img_resized = resize_image(final_img, target_size)
        video_writer.write(final_img_resized)

        video_writer.release()

print(f"Plots saved as PDF and videos created for each Sample_ID in {output_dir}")


## 03Plots_ Plot Neuron Maps based on Activity & Condition


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import cv2
import numpy as np
from matplotlib.colors import LinearSegmentedColormap

csv_file = UNIT_BURST_CSV
final_df = pd.read_csv(csv_file)

filtered_df = final_df[final_df['Mean Freq.'] >= 0.01]

filtered_df = filtered_df.copy()
filtered_df['burstiness'] = (
    filtered_df['Bursts Per Second'] *
    filtered_df['% of Spikes in Bursts'] *
    filtered_df['Mean Freq. in Burst'] *
    filtered_df['Mean Burst Duration']
)

output_dir = RESULTS_DIR / "03Plots_01_NeuronsMappedPosConditionActivity"
os.makedirs(output_dir, exist_ok=True)

video_frames = {}

min_size = 10
max_size = 100

orange_cmap = LinearSegmentedColormap.from_list("custom_orange", ["#FFD700","#FF4500"])

condition_order = ['Baseline', 'CBZ30uM', 'CBZ60uM', 'CBZ100uM', 'Washout']

for sample_id in filtered_df['Sample_ID'].unique():

    sample_df = filtered_df[filtered_df['Sample_ID'] == sample_id]

    max_burstiness = sample_df['burstiness'].max()
    filtered_df.loc[filtered_df['Sample_ID'] == sample_id, 'burstiness_normalized'] = (
    filtered_df.loc[filtered_df['Sample_ID'] == sample_id, 'burstiness'] / max_burstiness)

    video_frames[sample_id] = {}

    for condition in condition_order:

        condition_df = sample_df[sample_df['Condition'] == condition]
        if condition_df.empty:
            continue

        condition_df = condition_df.copy()
        condition_df['burstiness_normalized'] = (condition_df['burstiness'] / condition_df['burstiness'].max()) * 5

        x_positions = condition_df['x']
        y_positions = condition_df['y']
        neuron_sizes = condition_df['Mean Freq.'] * 3

        neuron_sizes = np.clip(neuron_sizes, min_size, max_size)

        burstiness_colors = condition_df['burstiness_normalized']

        plt.figure(figsize=(12, 6), facecolor='black')
        ax = plt.gca()
        ax.set_facecolor('black')

        scatter = plt.scatter(x_positions, y_positions, s=neuron_sizes,
                              c=burstiness_colors, cmap=orange_cmap,
                              edgecolor='none', alpha=0.8, vmin=0, vmax=1)

        cbar = plt.colorbar(scatter)
        cbar.set_label('Normalized Burstiness (0 to 1)', color='white')
        cbar.ax.yaxis.set_tick_params(color='white')
        plt.setp(plt.getp(cbar.ax.axes, 'yticklabels'), color='white')

        plt.title(f'Neuron Positions for Sample_ID: {sample_id} (Condition={condition})', color='white')
        plt.xlabel('X Position', color='white')
        plt.ylabel('Y Position', color='white')

        ax.tick_params(colors='white')

        plt.xlim(0, 4000)
        plt.ylim(0, 2200)

        rect = plt.Rectangle((0, 0), 4000, 2200, linewidth=2, edgecolor='white', facecolor='none')
        ax.add_patch(rect)

        image_output_path_png = os.path.join(output_dir, f'{sample_id}_condition_{condition}.png')
        plt.savefig(image_output_path_png, format='png', bbox_inches='tight', dpi=300)

        image_output_path_pdf = os.path.join(output_dir, f'{sample_id}_condition_{condition}.pdf')
        plt.savefig(image_output_path_pdf, format='pdf', bbox_inches='tight', dpi=300)

        video_frames[sample_id][condition] = image_output_path_png

        plt.close()

def resize_image(image, target_size):
    return cv2.resize(image, target_size, interpolation=cv2.INTER_LANCZOS4)

def interpolate_images(img1, img2, steps=10):
    interpolated_images = []
    for i in range(steps):
        alpha = i / float(steps)
        beta = 1.0 - alpha
        blended_img = cv2.addWeighted(img1, beta, img2, alpha, 0)
        interpolated_images.append(blended_img)
    return interpolated_images

for sample_id, condition_images in video_frames.items():
    video_output_path = os.path.join(output_dir, f'{sample_id}_plots_video.mp4')

    sorted_image_paths = [condition_images[condition] for condition in condition_order if condition in condition_images]

    if len(sorted_image_paths) == len(condition_order):
        first_image = cv2.imread(sorted_image_paths[0])
        height, width, _ = first_image.shape
        target_size = (width, height)

        video_writer = cv2.VideoWriter(video_output_path, cv2.VideoWriter_fourcc(*'mp4v'), 10, target_size)

        for i in range(len(sorted_image_paths) - 1):
            img1 = cv2.imread(sorted_image_paths[i])
            img2 = cv2.imread(sorted_image_paths[i + 1])

            img1_resized = resize_image(img1, target_size)
            img2_resized = resize_image(img2, target_size)

            video_writer.write(img1_resized)

            interpolated_frames = interpolate_images(img1_resized, img2_resized, steps=10)
            for frame in interpolated_frames:
                video_writer.write(frame)

        final_img = cv2.imread(sorted_image_paths[-1])
        video_writer.write(resize_image(final_img, target_size))

        video_writer.release()

print(f"Plots and videos created for each Sample_ID in {output_dir}")


## 04Plots_01_MEABasedBurst_01 condition ---- Plot & -ExcelExport-


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from scipy.stats import skew, kurtosis, mode

csv_file = UNIT_BURST_CSV
final_df = pd.read_csv(csv_file)

final_df = final_df[final_df['Sample_ID'] != 18574]

output_dir = RESULTS_DIR / "04Plots_01_MEABasedBurst_01"
os.makedirs(output_dir, exist_ok=True)

variables = ['Mean Freq.', 'Bursts Per Second', '% of Spikes in Bursts',
             'Mean Freq. in Burst', 'Mean Burst Duration']

condition_order = ['Baseline', 'CBZ30uM', 'CBZ60uM', 'CBZ100uM', 'Washout']

filtered_df = final_df[final_df['Mean Freq.'] >= 0.01]

sample_stats_per_condition = {}

for sample_id in filtered_df['Sample_ID'].unique():
    sample_df = filtered_df[filtered_df['Sample_ID'] == sample_id]
    sample_condition_stats = {}

    for condition in sample_df['Condition'].unique():
        condition_df = sample_df[sample_df['Condition'] == condition]

        condition_stats = {}
        for var in variables:
            values = condition_df[var]

            mean_val = values.mean()
            sdv_val = values.std()
            sem_val = sdv_val / np.sqrt(len(values))
            median_val = values.median()
            max_val = values.max()
            min_val = values.min()
            cv_val = sdv_val / mean_val if mean_val != 0 else np.nan
            variance_val = np.var(values)
            percentile_25 = np.percentile(values, 25)
            percentile_50 = np.percentile(values, 50)
            percentile_75 = np.percentile(values, 75)
            skewness_val = skew(values)
            kurtosis_val = kurtosis(values)
            mode_val = mode(values)[0][0] if isinstance(mode(values)[0], np.ndarray) else mode(values)[0]
            z_scores = (values - mean_val) / sdv_val
            avg_z_score = np.mean(z_scores)

            condition_stats[var] = {
                'mean': mean_val,
                'sdv': sdv_val,
                'sem': sem_val,
                'median': median_val,
                'max': max_val,
                'min': min_val,
                'cv': cv_val,
                'variance': variance_val,
                'percentile_25': percentile_25,
                'percentile_50': percentile_50,
                'percentile_75': percentile_75,
                'skewness': skewness_val,
                'kurtosis': kurtosis_val,
                'mode': mode_val,
                'average_z_score': avg_z_score
            }

        sample_condition_stats[condition] = condition_stats

    sample_stats_per_condition[sample_id] = sample_condition_stats

group_stats = {}
for group in filtered_df['Group'].unique():
    group_df = filtered_df[filtered_df['Group'] == group]

    group_avg_stats = {}
    for condition in group_df['Condition'].unique():
        samples_in_group_condition = [
            sample_stats_per_condition[sample_id][condition]
            for sample_id in group_df[group_df['Condition'] == condition]['Sample_ID'].unique()
        ]
        condition_group_stats = {}

        for var in variables:

            sample_means = [sample[var]['mean'] for sample in samples_in_group_condition]
            mean_val = np.mean(sample_means)
            sem_val = np.std(sample_means) / np.sqrt(len(sample_means))

            condition_group_stats[var] = {
                'mean': mean_val,
                'sem': sem_val,
                'sample_means': sample_means
            }

        group_avg_stats[condition] = condition_group_stats

    group_stats[group] = group_avg_stats

group_colors = {
    'AAV-Ctrl': '#1f77b4',
    'AAV-SMS': '#ff7f0e'
}

for var in variables:
    plt.figure(figsize=(10, 6))

    for group_idx, (group, group_data) in enumerate(group_stats.items()):
        conditions = [condition for condition in condition_order if condition in group_data]
        color = group_colors.get(group, 'black')

        boxplot_data = [group_data[condition][var]['sample_means'] for condition in conditions]

        x_positions = np.arange(len(conditions)) + group_idx * 0.2
        box = plt.boxplot(
            boxplot_data,
            positions=x_positions,
            widths=0.3,
            patch_artist=True,
            boxprops=dict(facecolor=color, edgecolor='black', linewidth=0.5),
            medianprops=dict(color='black', linewidth=1),
            whiskerprops=dict(color='black', linewidth=0.2),
            capprops=dict(color='black', linewidth=0.2),
            flierprops=dict(marker='o', color=color, markersize=5, alpha=0.5),
            zorder=1
        )

        means = [group_data[condition][var]['mean'] for condition in conditions]
        plt.plot(x_positions, means, marker='o', color=color,
                 label=f'{group} Mean', linewidth=1, linestyle='-',
                 markersize=6, markeredgecolor='black', zorder=2)

        for pos, sample_means in zip(x_positions, boxplot_data):
            jittered_x = pos + np.random.normal(0, 0.05, size=len(sample_means))
            plt.scatter(jittered_x, sample_means, color=color, edgecolor='black',
                        alpha=0.7, linewidth=0.2, zorder=3)

    plt.title(f'{var} (Box Plot with Scatter Points and Mean Lines) across Conditions')
    plt.xlabel('Condition')
    plt.ylabel(var)

    plt.legend(title="Groups", bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.xticks(ticks=np.arange(len(conditions)), labels=conditions)

    plot_output_path = os.path.join(output_dir, f'{var}_group_comparison_boxplot.pdf')
    plt.savefig(plot_output_path, format='pdf', bbox_inches='tight')
    plt.close()

output_excel_file = RESULTS_DIR / "04_01_MEAbasedBurstStat_01.xlsx"
with pd.ExcelWriter(output_excel_file) as writer:
    for var in variables:
        var_stats = []
        for sample_id, sample_stats in sample_stats_per_condition.items():
            for condition, condition_stats in sample_stats.items():
                metadata_row = filtered_df[filtered_df['Sample_ID'] == sample_id].iloc[0]

                var_stats.append({
                    'Group': metadata_row['Group'],
                    'Sample_ID': sample_id,
                    'Date': metadata_row['Date'],
                    'Condition': condition,
                    'mean': condition_stats[var]['mean'],
                    'sdv': condition_stats[var]['sdv'],
                    'sem': condition_stats[var]['sem'],
                    'median': condition_stats[var]['median'],
                    'max': condition_stats[var]['max'],
                    'min': condition_stats[var]['min'],
                    'cv': condition_stats[var]['cv'],
                    'variance': condition_stats[var]['variance'],
                    'percentile_25': condition_stats[var]['percentile_25'],
                    'percentile_50': condition_stats[var]['percentile_50'],
                    'percentile_75': condition_stats[var]['percentile_75'],
                    'skewness': condition_stats[var]['skewness'],
                    'kurtosis': condition_stats[var]['kurtosis'],
                    'mode': condition_stats[var]['mode'],
                    'average_z_score': condition_stats[var]['average_z_score']
                })

        var_df = pd.DataFrame(var_stats)
        var_df.to_excel(writer, sheet_name=var, index=False)

print("Box plots, including all statistics, and the statistics have been exported to Excel.")


## Selected_Conditions_04Plots_01_MEABasedBurst_01 20241127 ----Plot & ---ExcelExport---

CBZ60uM excluded


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from scipy.stats import skew, kurtosis, mode

csv_file = UNIT_BURST_CSV
final_df = pd.read_csv(csv_file)

final_df = final_df[(final_df['Sample_ID'] != 18574) & (final_df['Condition'] != 'CBZ60uM') & (final_df['Condition'] != 'Washout')]

output_dir = RESULTS_DIR / "Selected_04Plots_01_MEABasedBurst_01"
os.makedirs(output_dir, exist_ok=True)

variables = ['Mean Freq.', 'Bursts Per Second', '% of Spikes in Bursts',
             'Mean Freq. in Burst', 'Mean Burst Duration']

condition_order = ['Baseline', 'CBZ30uM', 'CBZ100uM']

filtered_df = final_df[final_df['Mean Freq.'] >= 0.01]

sample_stats_per_condition = {}

for sample_id in filtered_df['Sample_ID'].unique():
    sample_df = filtered_df[filtered_df['Sample_ID'] == sample_id]
    sample_condition_stats = {}

    for condition in sample_df['Condition'].unique():
        condition_df = sample_df[sample_df['Condition'] == condition]

        condition_stats = {}
        for var in variables:
            values = condition_df[var]

            mean_val = values.mean()
            sdv_val = values.std()
            sem_val = sdv_val / np.sqrt(len(values))
            median_val = values.median()
            max_val = values.max()
            min_val = values.min()
            cv_val = sdv_val / mean_val if mean_val != 0 else np.nan
            variance_val = np.var(values)
            percentile_25 = np.percentile(values, 25)
            percentile_50 = np.percentile(values, 50)
            percentile_75 = np.percentile(values, 75)
            skewness_val = skew(values)
            kurtosis_val = kurtosis(values)
            mode_val = mode(values)[0][0] if isinstance(mode(values)[0], np.ndarray) else mode(values)[0]
            z_scores = (values - mean_val) / sdv_val
            avg_z_score = np.mean(z_scores)

            condition_stats[var] = {
                'mean': mean_val,
                'sdv': sdv_val,
                'sem': sem_val,
                'median': median_val,
                'max': max_val,
                'min': min_val,
                'cv': cv_val,
                'variance': variance_val,
                'percentile_25': percentile_25,
                'percentile_50': percentile_50,
                'percentile_75': percentile_75,
                'skewness': skewness_val,
                'kurtosis': kurtosis_val,
                'mode': mode_val,
                'average_z_score': avg_z_score
            }

        sample_condition_stats[condition] = condition_stats

    sample_stats_per_condition[sample_id] = sample_condition_stats

group_stats = {}
for group in filtered_df['Group'].unique():
    group_df = filtered_df[filtered_df['Group'] == group]

    group_avg_stats = {}
    for condition in group_df['Condition'].unique():
        samples_in_group_condition = [
            sample_stats_per_condition[sample_id][condition]
            for sample_id in group_df[group_df['Condition'] == condition]['Sample_ID'].unique()
        ]
        condition_group_stats = {}

        for var in variables:

            sample_means = [sample[var]['mean'] for sample in samples_in_group_condition]
            mean_val = np.mean(sample_means)
            sem_val = np.std(sample_means) / np.sqrt(len(sample_means))

            condition_group_stats[var] = {
                'mean': mean_val,
                'sem': sem_val,
                'sample_means': sample_means
            }

        group_avg_stats[condition] = condition_group_stats

    group_stats[group] = group_avg_stats

group_colors = {
    'AAV-Ctrl': '#1f77b4',
    'AAV-SMS': '#ff7f0e'
}

for var in variables:
    plt.figure(figsize=(4, 6))

    global_max = 0

    for group_idx, (group, group_data) in enumerate(group_stats.items()):
        conditions = [condition for condition in condition_order if condition in group_data]
        color = group_colors.get(group, 'black')

        boxplot_data = [group_data[condition][var]['sample_means'] for condition in conditions]

        group_max = max([max(data) for data in boxplot_data if len(data) > 0])
        global_max = max(global_max, group_max)

        x_positions = np.arange(len(conditions)) + group_idx * 0.25
        box = plt.boxplot(
            boxplot_data,
            positions=x_positions,
            widths=0.25,
            patch_artist=True,
            boxprops=dict(facecolor=color, edgecolor='black', linewidth=0.7, alpha=0.7),
            medianprops=dict(color='black', linewidth=1),
            whiskerprops=dict(color='black', linewidth=0.7),
            capprops=dict(color='black', linewidth=0.7),
            flierprops=dict(marker='o', color=color, markersize=5, alpha=0.5),
            zorder=1
        )

        means = [group_data[condition][var]['mean'] for condition in conditions]
        plt.plot(x_positions, means, marker='o', color=color,
                 label=f'{group} Mean', linewidth=1.5, linestyle='-',
                 markersize=8, markeredgecolor='black', zorder=2)

        for pos, sample_means in zip(x_positions, boxplot_data):
            jittered_x = pos + np.random.normal(0, 0.05, size=len(sample_means))
            plt.scatter(jittered_x, sample_means, color=color, edgecolor='black',
                        alpha=0.9, linewidth=0.5, zorder=3, s=40)

    max_y = global_max * 1.3
    plt.ylim(0, max_y)

    plt.title(f'{var} (Box Plot with Scatter Points and Mean Lines) across Conditions',
            fontdict={'fontsize': 14, 'fontweight': 'bold', 'family': 'serif'})

    plt.xlabel('Condition', fontdict={'fontsize': 14, 'fontweight': 'medium', 'family': 'sans-serif'})

    plt.ylabel(var, fontdict={'fontsize': 14, 'fontweight': 'medium', 'family': 'sans-serif'})

    plt.legend(title="Groups", bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=14, title_fontsize=14)

    plt.xticks(ticks=np.arange(len(conditions)), labels=conditions, fontsize=14)

    plot_output_path = os.path.join(output_dir, f'{var}_Boxplot.pdf')
    plt.savefig(plot_output_path, format='pdf', bbox_inches='tight')
    plt.close()

output_excel_file = RESULTS_DIR / "04_01_SelectedMEAbasedBurstStat_01.xlsx"
with pd.ExcelWriter(output_excel_file) as writer:
    for var in variables:
        var_stats = []
        for sample_id, sample_stats in sample_stats_per_condition.items():
            for condition, condition_stats in sample_stats.items():
                metadata_row = filtered_df[filtered_df['Sample_ID'] == sample_id].iloc[0]

                var_stats.append({
                    'Group': metadata_row['Group'],
                    'Sample_ID': sample_id,
                    'Date': metadata_row['Date'],
                    'Condition': condition,
                    'mean': condition_stats[var]['mean'],
                    'sdv': condition_stats[var]['sdv'],
                    'sem': condition_stats[var]['sem'],
                    'median': condition_stats[var]['median'],
                    'max': condition_stats[var]['max'],
                    'min': condition_stats[var]['min'],
                    'cv': condition_stats[var]['cv'],
                    'variance': condition_stats[var]['variance'],
                    'percentile_25': condition_stats[var]['percentile_25'],
                    'percentile_50': condition_stats[var]['percentile_50'],
                    'percentile_75': condition_stats[var]['percentile_75'],
                    'skewness': condition_stats[var]['skewness'],
                    'kurtosis': condition_stats[var]['kurtosis'],
                    'mode': condition_stats[var]['mode'],
                    'average_z_score': condition_stats[var]['average_z_score']
                })

        var_df = pd.DataFrame(var_stats)
        var_df.to_excel(writer, sheet_name=var, index=False)

print("Box plots, including all statistics, and the statistics have been exported to Excel.")


## 04Plots_02_Normalized_MEABasedBurst_01 condition ----  Plot & ---ExcelExport---


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from scipy.stats import skew, kurtosis, mode

csv_file = UNIT_BURST_CSV
final_df = pd.read_csv(csv_file)

final_df = final_df[final_df['Sample_ID'] != 18574]

output_dir = RESULTS_DIR / "04Plots_02_NormMEABasedBurst_01"
os.makedirs(output_dir, exist_ok=True)

variables = ['Mean Freq.', 'Bursts Per Second', '% of Spikes in Bursts',
             'Mean Freq. in Burst', 'Mean Burst Duration']

condition_order = ['Baseline', 'CBZ30uM', 'CBZ60uM', 'CBZ100uM', 'Washout']

filtered_df = final_df[final_df['Mean Freq.'] >= 0.01]

sample_stats_per_condition = {}

for sample_id in filtered_df['Sample_ID'].unique():
    sample_df = filtered_df[filtered_df['Sample_ID'] == sample_id]
    sample_condition_stats = {}

    baseline_df = sample_df[sample_df['Condition'] == 'Baseline']
    baseline_values = baseline_df[variables].mean().replace(0, 0.001)

    for condition in sample_df['Condition'].unique():
        condition_df = sample_df[sample_df['Condition'] == condition]

        condition_df.loc[:, variables] = condition_df[variables].div(baseline_values, axis=1)

        condition_stats = {}
        for var in variables:
            values = condition_df[var]

            mean_val = values.mean()
            sdv_val = values.std()
            sem_val = sdv_val / np.sqrt(len(values))
            median_val = values.median()
            max_val = values.max()
            min_val = values.min()
            cv_val = sdv_val / mean_val if mean_val != 0 else np.nan
            variance_val = np.var(values)
            percentile_25 = np.percentile(values, 25)
            percentile_50 = np.percentile(values, 50)
            percentile_75 = np.percentile(values, 75)
            skewness_val = skew(values)
            kurtosis_val = kurtosis(values)
            mode_val = mode(values)[0][0] if isinstance(mode(values)[0], np.ndarray) else mode(values)[0]
            z_scores = (values - mean_val) / sdv_val
            avg_z_score = np.mean(z_scores)

            condition_stats[var] = {
                'mean': mean_val,
                'sdv': sdv_val,
                'sem': sem_val,
                'median': median_val,
                'max': max_val,
                'min': min_val,
                'cv': cv_val,
                'variance': variance_val,
                'percentile_25': percentile_25,
                'percentile_50': percentile_50,
                'percentile_75': percentile_75,
                'skewness': skewness_val,
                'kurtosis': kurtosis_val,
                'mode': mode_val,
                'average_z_score': avg_z_score
            }

        sample_condition_stats[condition] = condition_stats

    sample_stats_per_condition[sample_id] = sample_condition_stats

group_stats = {}
for group in filtered_df['Group'].unique():
    group_df = filtered_df[filtered_df['Group'] == group]

    group_avg_stats = {}
    for condition in group_df['Condition'].unique():
        samples_in_group_condition = [
            sample_stats_per_condition[sample_id][condition]
            for sample_id in group_df[group_df['Condition'] == condition]['Sample_ID'].unique()
        ]
        condition_group_stats = {}

        for var in variables:

            sample_means = [sample[var]['mean'] for sample in samples_in_group_condition]
            mean_val = np.mean(sample_means)
            sem_val = np.std(sample_means) / np.sqrt(len(sample_means))

            condition_group_stats[var] = {
                'mean': mean_val,
                'sem': sem_val,
                'sample_means': sample_means
            }

        group_avg_stats[condition] = condition_group_stats

    group_stats[group] = group_avg_stats

group_colors = {
    'AAV-Ctrl': '#1f77b4',
    'AAV-SMS': '#ff7f0e'
}

for var in variables:
    plt.figure(figsize=(10, 6))

    for group_idx, (group, group_data) in enumerate(group_stats.items()):
        conditions = [condition for condition in condition_order if condition in group_data]
        color = group_colors.get(group, 'black')

        boxplot_data = [group_data[condition][var]['sample_means'] for condition in conditions]

        x_positions = np.arange(len(conditions)) + group_idx * 0.2
        box = plt.boxplot(
            boxplot_data,
            positions=x_positions,
            widths=0.3,
            patch_artist=True,
            boxprops=dict(facecolor=color, edgecolor='black', linewidth=0.5),
            medianprops=dict(color='black', linewidth=1),
            whiskerprops=dict(color='black', linewidth=0.2),
            capprops=dict(color='black', linewidth=0.2),
            flierprops=dict(marker='o', color=color, markersize=5, alpha=0.5),
            zorder=1
        )

        means = [group_data[condition][var]['mean'] for condition in conditions]
        plt.plot(x_positions, means, marker='o', color=color,
                 label=f'{group} Mean', linewidth=1, linestyle='-',
                 markersize=6, markeredgecolor='black', zorder=2)

        for pos, sample_means in zip(x_positions, boxplot_data):
            jittered_x = pos + np.random.normal(0, 0.05, size=len(sample_means))
            plt.scatter(jittered_x, sample_means, color=color, edgecolor='black',
                        alpha=0.7, linewidth=0.2, zorder=3)

    plt.title(f'{var} (Box Plot with Scatter Points and Mean Lines) across Conditions')
    plt.xlabel('Condition')
    plt.ylabel(var)

    plt.legend(title="Groups", bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.xticks(ticks=np.arange(len(conditions)), labels=conditions)

    plot_output_path = os.path.join(output_dir, f'{var}_group_comparison_boxplot.pdf')
    plt.savefig(plot_output_path, format='pdf', bbox_inches='tight')
    plt.close()

output_excel_file = RESULTS_DIR / "04_02_NormMEAbasedBurstStat_01.xlsx"
with pd.ExcelWriter(output_excel_file) as writer:
    for var in variables:
        var_stats = []
        for sample_id, sample_stats in sample_stats_per_condition.items():
            for condition, condition_stats in sample_stats.items():
                metadata_row = filtered_df[filtered_df['Sample_ID'] == sample_id].iloc[0]

                var_stats.append({
                    'Group': metadata_row['Group'],
                    'Sample_ID': sample_id,
                    'Date': metadata_row['Date'],
                    'Condition': condition,
                    'mean': condition_stats[var]['mean'],
                    'sdv': condition_stats[var]['sdv'],
                    'sem': condition_stats[var]['sem'],
                    'median': condition_stats[var]['median'],
                    'max': condition_stats[var]['max'],
                    'min': condition_stats[var]['min'],
                    'cv': condition_stats[var]['cv'],
                    'variance': condition_stats[var]['variance'],
                    'percentile_25': condition_stats[var]['percentile_25'],
                    'percentile_50': condition_stats[var]['percentile_50'],
                    'percentile_75': condition_stats[var]['percentile_75'],
                    'skewness': condition_stats[var]['skewness'],
                    'kurtosis': condition_stats[var]['kurtosis'],
                    'mode': condition_stats[var]['mode'],
                    'average_z_score': condition_stats[var]['average_z_score']
                })

        var_df = pd.DataFrame(var_stats)
        var_df.to_excel(writer, sheet_name=var, index=False)

print("Box plots, including all statistics, and the statistics have been exported to Excel.")


## Normalize the individual neuronal burst data to the mean of that Sample_ID in Baseline


In [ ]:
import pandas as pd
import numpy as np

input_csv_file = UNIT_BURST_CSV
df = pd.read_csv(input_csv_file)

variables = ['Mean Freq.', 'Bursts Per Second', '% of Spikes in Bursts',
             'Mean Freq. in Burst', 'Mean Burst Duration']

baseline_df = df[df['Condition'] == 'Baseline']

baseline_stats = baseline_df.groupby('Sample_ID')[variables].agg(['mean', 'std']).replace(0, 0.001)

baseline_stats.columns = [f'{col[0]}_{col[1]}' for col in baseline_stats.columns]

df = df.merge(baseline_stats, on='Sample_ID', suffixes=('', '_baseline'))

normalized_df = df.copy()

for var in variables:

    normalized_column = f'{var}'

    normalized_df[normalized_column] = (df[var] - df[f'{var}_mean']) / df[f'{var}_std']

columns_to_keep = ['unit_id', 'x', 'y', 'z', 'Condition', 'File Name', 'Variable', 'Sample_ID', 'Group'] + [f'{var}' for var in variables]

normalized_df = normalized_df[columns_to_keep]

output_csv_file = NORMALIZED_UNIT_BURST_CSV
normalized_df.to_csv(output_csv_file, index=False)

print("Normalized data has been successfully exported to Excel.")


## Plot RH, NetBurst, Raster For specific sampels


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

output_pickle_path = RATE_HISTOGRAM_PKL
loaded_df = pd.read_pickle(output_pickle_path)

sample_df = loaded_df[loaded_df['File Name'] == '20240814_18389_01']
sample_df.loc[:, 'unit_id'] = sample_df['unit_id'].astype(int)
neuron_ids = [0, 3, 6, 43, 56, 89, 90, 93, 95, 102, 116, 117, 120, 150, 152]
selected_neurons_df = sample_df[sample_df['unit_id'].isin(neuron_ids)]
time_range_start = 100
time_range_end = 300

plt.figure(figsize=(6.4, 1))
point_size = 0.5
number_of_neurons = len(neuron_ids)
for i, (index, row) in enumerate(selected_neurons_df.iterrows()):
    neuron_id = row['unit_id']
    neuron_timeseries = row['Timeseries']
    filtered_timeseries = [t for t in neuron_timeseries if time_range_start <= t <= time_range_end]
    plt.scatter(filtered_timeseries, [i] * len(filtered_timeseries),
                s=point_size, alpha=0.5, color='gray')
plt.xlim([time_range_start, time_range_end])
plt.title('Raster Plot of Neurons Timeseries Data (100 to 200 seconds)')
plt.xlabel('Time (seconds)')
plt.ylabel('Neuron Index')
plt.yticks(ticks=range(len(neuron_ids)), labels=[f'Neuron {nid}' for nid in neuron_ids])
plt.show()

fig, (ax_burst, ax_heatmap) = plt.subplots(2, 1, figsize=(8, 6),
                                           gridspec_kw={'height_ratios': [1, 3], 'hspace': 0.1},
                                           constrained_layout=False)

window_size = 3
rate_histo_matrix = []

for i, (index, row) in enumerate(selected_neurons_df.iterrows()):
    neuron_id = row['unit_id']
    rate_histo = row['Rate_Histo']
    left_bins = row['Left_Bins']
    left_bins = np.array(left_bins)

    time_mask = (left_bins >= time_range_start) & (left_bins <= time_range_end)
    filtered_rate_histo = np.array(rate_histo)[time_mask]

    rate_histo_matrix.append(filtered_rate_histo)

rate_histo_matrix = np.array(rate_histo_matrix)

network_burst = np.mean(rate_histo_matrix, axis=0)

network_burst_timepoints = np.linspace(time_range_start, time_range_end, len(network_burst))

ax_burst.plot(network_burst_timepoints, network_burst, color='red', linewidth=0.5, label='Network Burst')
ax_burst.set_ylabel('Network Burst')
ax_burst.set_xlim([time_range_start, time_range_end])
ax_burst.legend(loc='upper left', bbox_to_anchor=(1, 1))
ax_burst.xaxis.set_visible(False)

im = ax_heatmap.imshow(rate_histo_matrix, aspect='auto', cmap='hot', origin='lower',
                       extent=[time_range_start, time_range_end, 0, len(neuron_ids)])

ax_heatmap.set_title('Smoothed Rate Histogram Heatmap (100 to 200 seconds)')
ax_heatmap.set_xlabel('Time (seconds)')
ax_heatmap.set_ylabel('Neuron ID')
ax_heatmap.set_yticks(np.arange(len(neuron_ids)))
ax_heatmap.set_yticklabels([f'Neuron {nid}' for nid in neuron_ids])
cbar = plt.colorbar(im, ax=ax_heatmap)
cbar.set_label('Firing Rate')

pos1 = ax_burst.get_position()
pos2 = ax_heatmap.get_position()

ax_burst.set_position([pos2.x0, pos1.y0, pos2.width, pos1.height])
ax_heatmap.set_position([pos2.x0, pos2.y0, pos2.width, pos2.height])
plt.show()

print("\nFiltered DataFrame for selected neurons:")
print(selected_neurons_df)


## 05_Plots_01_NetworkBurstPeaks, threshold line, Normalized RH and rasterplot of Ts __ Save as PDF


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
from matplotlib.backends.backend_pdf import PdfPages
from scipy.signal import find_peaks, peak_widths

output_pickle_path = RATE_HISTOGRAM_PKL
loaded_df = pd.read_pickle(output_pickle_path)
output_dir = RESULTS_DIR / "05Plots_01_NetBurstsRasterRH_600s_01"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

time_range_start = 0
time_range_end = 600

for file_name in loaded_df['File Name'].unique():
    sample_df = loaded_df[loaded_df['File Name'] == file_name]
    sample_df.loc[:, 'unit_id'] = sample_df['unit_id'].astype(int)

    if len(sample_df['unit_id'].unique()) > 300:
        selected_unit_ids = np.random.choice(sample_df['unit_id'].unique(), size=300, replace=False)
    else:
        selected_unit_ids = sample_df['unit_id'].unique()

    selected_neurons_df = sample_df[sample_df['unit_id'].isin(selected_unit_ids)]

    pdf_file_path = os.path.join(output_dir, f'{file_name}_combined_plots.pdf')
    with PdfPages(pdf_file_path) as pdf:

        plt.figure(figsize=(6.4, 6))
        point_size = 0.1
        number_of_neurons = len(selected_unit_ids)

        for i, (index, row) in enumerate(selected_neurons_df.iterrows()):
            neuron_id = row['unit_id']
            neuron_timeseries = row['Timeseries']
            filtered_timeseries = [t for t in neuron_timeseries if time_range_start <= t <= time_range_end]
            plt.scatter(filtered_timeseries, [i] * len(filtered_timeseries),
                        s=point_size, alpha=0.1, color='gray')

        plt.xlim([time_range_start, time_range_end])
        plt.title(f'Raster Plot of Neurons Timeseries Data ({file_name})')
        plt.xlabel('Time (seconds)')
        plt.ylabel('Neuron Index')
        plt.yticks(ticks=range(number_of_neurons), labels=[f'Neuron {nid}' for nid in selected_unit_ids])

        pdf.savefig()
        plt.close()

        fig, (ax_burst, ax_heatmap) = plt.subplots(2, 1, figsize=(8, 6),
                                                   gridspec_kw={'height_ratios': [1, 3], 'hspace': 0.1},
                                                   constrained_layout=False)

        rate_histo_matrix = []

        frequency_domain_counts = np.zeros(101)

        for i, (index, row) in enumerate(selected_neurons_df.iterrows()):
            neuron_id = row['unit_id']
            rate_histo = row['Rate_Histo']
            left_bins = row['Left_Bins']
            left_bins = np.array(left_bins)

            time_mask = (left_bins >= time_range_start) & (left_bins <= time_range_end)
            filtered_rate_histo = np.array(rate_histo)[time_mask]
            rate_histo_matrix.append(filtered_rate_histo)

            for frequency in filtered_rate_histo:
                if 0 <= frequency <= 100:
                    frequency_domain_counts[int(frequency)] += 1

        total_bins = np.sum(frequency_domain_counts)
        rate_histo_matrix = np.array(rate_histo_matrix)

        max_value = np.nanmax(rate_histo_matrix)

        if max_value > 0:
            normalized_rate_histo_matrix = rate_histo_matrix / max_value
        else:
            normalized_rate_histo_matrix = rate_histo_matrix / 1

        network_burst = np.mean(rate_histo_matrix, axis=0)
        network_burst_timepoints = np.linspace(time_range_start, time_range_end, len(network_burst))

        mean_burst = np.mean(network_burst)
        std_burst = np.std(network_burst)
        threshold = (0.5/mean_burst) + (mean_burst + 1.5 * std_burst)

        min_distance = 20

        peaks, _ = find_peaks(network_burst, height=threshold, distance=min_distance)

        peak_amplitudes = network_burst[peaks]
        peak_widths_values, width_heights, left_ips, right_ips = peak_widths(network_burst, peaks, rel_height=0.5)
        peak_intervals = np.diff(network_burst_timepoints[peaks])
        peak_frequencies = 1 / peak_intervals

        ax_burst.plot(network_burst_timepoints, network_burst, color='red', linewidth=0.5, label='Network Burst')

        ax_burst.axhline(y=threshold, color='black', linestyle='--', linewidth=1, label='Threshold')

        ax_burst.scatter(network_burst_timepoints[peaks], network_burst[peaks],
                         facecolor='gray', edgecolor='black', s=30, zorder=5, label='Detected Peaks')

        ax_burst.set_ylabel('Network Burst')
        ax_burst.set_xlim([time_range_start, time_range_end])
        ax_burst.legend(loc='upper left', bbox_to_anchor=(1, 1))
        ax_burst.xaxis.set_visible(False)

        im = ax_heatmap.imshow(normalized_rate_histo_matrix, aspect='auto', cmap='hot', origin='lower',
                               extent=[time_range_start, time_range_end, 0, len(selected_unit_ids)])

        ax_heatmap.set_title(f'Smoothed Rate Histogram Heatmap ({file_name})')
        ax_heatmap.set_xlabel('Time (seconds)')
        ax_heatmap.set_ylabel('Neuron ID')
        ax_heatmap.set_yticks(np.arange(len(selected_unit_ids)))
        ax_heatmap.set_yticklabels([f'Neuron {nid}' for nid in selected_unit_ids])

        cbar = plt.colorbar(im, ax=ax_heatmap)
        cbar.set_label('Normalized Firing Rate')

        pos1 = ax_burst.get_position()
        pos2 = ax_heatmap.get_position()

        ax_burst.set_position([pos2.x0, pos1.y0, pos2.width, pos1.height])
        ax_heatmap.set_position([pos2.x0, pos2.y0, pos2.width, pos2.height])

        pdf.savefig()
        plt.close()


## 05_01_Plots_NetworkburstPeaks, threshold line, normalized heatmap of RH and rasterplot of Ts __ save as PNG


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
from scipy.signal import find_peaks, peak_widths

output_pickle_path = RATE_HISTOGRAM_PKL
loaded_df = pd.read_pickle(output_pickle_path)
output_dir = RESULTS_DIR / "05Plots_01_NetBurstsRasterRH_600s_01"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

time_range_start = 0
time_range_end = 600

for file_name in loaded_df['File Name'].unique():
    sample_df = loaded_df[loaded_df['File Name'] == file_name]
    sample_df.loc[:, 'unit_id'] = sample_df['unit_id'].astype(int)

    if len(sample_df['unit_id'].unique()) > 300:
        selected_unit_ids = np.random.choice(sample_df['unit_id'].unique(), size=300, replace=False)
    else:
        selected_unit_ids = sample_df['unit_id'].unique()

    selected_neurons_df = sample_df[sample_df['unit_id'].isin(selected_unit_ids)]

    plt.figure(figsize=(6.4, 6))
    point_size = 0.1
    number_of_neurons = len(selected_unit_ids)

    for i, (index, row) in enumerate(selected_neurons_df.iterrows()):
        neuron_id = row['unit_id']
        neuron_timeseries = row['Timeseries']
        filtered_timeseries = [t for t in neuron_timeseries if time_range_start <= t <= time_range_end]
        plt.scatter(filtered_timeseries, [i] * len(filtered_timeseries),
                    s=point_size, alpha=0.1, color='gray')

    plt.xlim([time_range_start, time_range_end])
    plt.title(f'Raster Plot of Neurons Timeseries Data ({file_name})')
    plt.xlabel('Time (seconds)')
    plt.ylabel('Neuron Index')
    plt.yticks(ticks=range(number_of_neurons), labels=[f'Neuron {nid}' for nid in selected_unit_ids])

    raster_plot_path = os.path.join(output_dir, f'{file_name}_raster_plot.png')
    plt.savefig(raster_plot_path, dpi=600)
    plt.close()

    fig, (ax_burst, ax_heatmap) = plt.subplots(2, 1, figsize=(8, 6),
                                               gridspec_kw={'height_ratios': [1, 3], 'hspace': 0.1},
                                               constrained_layout=False)

    rate_histo_matrix = []

    frequency_domain_counts = np.zeros(101)

    for i, (index, row) in enumerate(selected_neurons_df.iterrows()):
        neuron_id = row['unit_id']
        rate_histo = row['Rate_Histo']
        left_bins = row['Left_Bins']
        left_bins = np.array(left_bins)

        time_mask = (left_bins >= time_range_start) & (left_bins <= time_range_end)
        filtered_rate_histo = np.array(rate_histo)[time_mask]

        rate_histo_matrix.append(filtered_rate_histo)

        for frequency in filtered_rate_histo:
            if 0 <= frequency <= 100:
                frequency_domain_counts[int(frequency)] += 1

    total_bins = np.sum(frequency_domain_counts)

    rate_histo_matrix = np.array(rate_histo_matrix)

    max_value = np.nanmax(rate_histo_matrix)

    if max_value > 0:
        normalized_rate_histo_matrix = rate_histo_matrix / max_value
    else:
        normalized_rate_histo_matrix = rate_histo_matrix / 1

    network_burst = np.mean(rate_histo_matrix, axis=0)
    network_burst_timepoints = np.linspace(time_range_start, time_range_end, len(network_burst))

    mean_burst = np.mean(network_burst)
    std_burst = np.std(network_burst)
    threshold = (0.5/mean_burst) + (mean_burst + 1.5 * std_burst)

    min_distance = 20

    peaks, _ = find_peaks(network_burst, height=threshold, distance=min_distance)

    peak_amplitudes = network_burst[peaks]
    peak_widths_values, width_heights, left_ips, right_ips = peak_widths(network_burst, peaks, rel_height=0.5)
    peak_intervals = np.diff(network_burst_timepoints[peaks])
    peak_frequencies = 1 / peak_intervals

    ax_burst.plot(network_burst_timepoints, network_burst, color='red', linewidth=0.5, label='Network Burst')

    ax_burst.axhline(y=threshold, color='black', linestyle='--', linewidth=1, label='Threshold')

    ax_burst.scatter(network_burst_timepoints[peaks], network_burst[peaks],
                     facecolor='gray', edgecolor='black', s=30, zorder=5, label='Detected Peaks')

    ax_burst.set_ylabel('Network Burst')
    ax_burst.set_xlim([time_range_start, time_range_end])
    ax_burst.legend(loc='upper left', bbox_to_anchor=(1, 1))
    ax_burst.xaxis.set_visible(False)

    im = ax_heatmap.imshow(normalized_rate_histo_matrix, aspect='auto', cmap='hot', origin='lower',
                           extent=[time_range_start, time_range_end, 0, len(selected_unit_ids)])

    ax_heatmap.set_title(f'Smoothed Rate Histogram Heatmap ({file_name})')
    ax_heatmap.set_xlabel('Time (seconds)')
    ax_heatmap.set_ylabel('Neuron ID')
    ax_heatmap.set_yticks(np.arange(len(selected_unit_ids)))
    ax_heatmap.set_yticklabels([f'Neuron {nid}' for nid in selected_unit_ids])

    cbar = plt.colorbar(im, ax=ax_heatmap)
    cbar.set_label('Normalized Firing Rate')

    pos1 = ax_burst.get_position()
    pos2 = ax_heatmap.get_position()

    ax_burst.set_position([pos2.x0, pos1.y0, pos2.width, pos1.height])
    ax_heatmap.set_position([pos2.x0, pos2.y0, pos2.width, pos2.height])

    combined_plot_path = os.path.join(output_dir, f'{file_name}_combined_plot.png')
    plt.savefig(combined_plot_path, dpi=600)
    plt.close()


## 05_01_Measure and save the NetworkBurstParams and FrequencyBandPercentages

- Amp,Freq,Width of peaks as list and Freq-binPercentages as Dictionary. SynchIndex as values (mean per sample)

- Amp, Freq and width stats were saved separatly as dictionary (e.g. Mean: , SDV: ,)..


In [ ]:
import pandas as pd
import numpy as np
from scipy.signal import find_peaks, peak_widths
from scipy.stats import sem
import xlsxwriter

output_pickle_path = RATE_HISTOGRAM_PKL
output_excel_path = NETWORK_BURST_XLSX
loaded_df = pd.read_pickle(output_pickle_path)

time_range_start = 0
time_range_end = 600

condition_order = ['Baseline', 'CBZ30uM', 'CBZ60uM', 'CBZ100uM', 'Washout']

loaded_df['Condition'] = pd.Categorical(loaded_df['Condition'], categories=condition_order, ordered=True)
loaded_df = loaded_df.sort_values(by='Condition')

results_list = []
for condition in condition_order:
    condition_df = loaded_df[loaded_df['Condition'] == condition]

    for file_name in condition_df['File Name'].unique():
        sample_df = condition_df[condition_df['File Name'] == file_name]
        group = sample_df['Group'].iloc[0]
        sample_id = sample_df['Sample_ID'].iloc[0]
        date = sample_df['Date'].iloc[0]
        condition = sample_df['Condition'].iloc[0]
        post_sms = sample_df['post_SMS'].iloc[0]

        rate_histo_matrix = []
        for i, row in sample_df.iterrows():
            rate_histo = np.array(row['Rate_Histo'])
            left_bins = np.array(row['Left_Bins'])

            time_mask = (left_bins >= time_range_start) & (left_bins <= time_range_end)
            filtered_rate_histo = rate_histo[time_mask]
            rate_histo_matrix.append(filtered_rate_histo)

        rate_histo_matrix = np.array(rate_histo_matrix)

        network_burst = np.mean(rate_histo_matrix, axis=0)

        mean_burst = np.mean(network_burst)
        std_burst = np.std(network_burst)
        threshold = (0.5/mean_burst) + (mean_burst + 1.5 * std_burst)

        min_distance = 20

        peaks, _ = find_peaks(network_burst, height=threshold, distance=min_distance)

        peak_amplitudes = network_burst[peaks]
        peak_widths_result = peak_widths(network_burst, peaks, rel_height=0.5)[0] * 0.1
        if len(peaks) > 1:
            peak_intervals = np.diff(peaks)
            peak_frequencies = 1 / (peak_intervals * 0.1)
        else:
            peak_frequencies = np.array([])

        def calculate_stats(data):
            return {
                'Mean': np.mean(data),
                'SD': np.std(data),
                'SEM': sem(data) if len(data) > 1 else None,
                'Max': np.max(data),
                'Min': np.min(data),
                'Median': np.median(data)
            }

        peak_amplitude_stats = calculate_stats(peak_amplitudes) if len(peak_amplitudes) > 0 else {}
        peak_frequency_stats = calculate_stats(peak_frequencies) if len(peak_frequencies) > 0 else {}
        peak_width_stats = calculate_stats(peak_widths_result) if len(peak_widths_result) > 0 else {}

        frequency_domain_counts = np.zeros(101)
        total_bins = 0

        for neuron_rates in rate_histo_matrix:
            for rate in neuron_rates:
                if 0 <= rate <= 100:
                    frequency_domain_counts[int(rate)] += 1
            total_bins += len(neuron_rates)

        frequency_percentages = [(count / total_bins) * 100 if total_bins > 0 else 0 for count in frequency_domain_counts]
        frequency_dict = {f'{i}Hz': f'{percentage:.2f}%' for i, percentage in enumerate(frequency_percentages)}

        if rate_histo_matrix.shape[0] > 1:

            variable_neurons = rate_histo_matrix[np.std(rate_histo_matrix, axis=1) > 0]

            if variable_neurons.shape[0] > 1:
                correlation_matrix = np.corrcoef(variable_neurons)

                np.nan_to_num(correlation_matrix, copy=False)
                upper_triangle_indices = np.triu_indices_from(correlation_matrix, k=1)
                synchronicity_index = np.mean(correlation_matrix[upper_triangle_indices])
            else:
                synchronicity_index = None
        else:
            synchronicity_index = None

        results_list.append({
            'Group': group,
            'File Name': file_name,
            'Sample_ID': sample_id,
            'Date': date,
            'Condition': condition,
            'post_SMS': post_sms,
            'Num_Peaks': len(peaks),
            'Peak_Amplitudes': peak_amplitudes.tolist(),
            'Peak_Amplitude_Stats': peak_amplitude_stats,
            'Peak_Frequencies': peak_frequencies.tolist(),
            'Peak_Frequency_Stats': peak_frequency_stats,
            'Peak_Widths': peak_widths_result.tolist(),
            'Peak_Width_Stats': peak_width_stats,
            'Synchronicity_Index': synchronicity_index,
            'Freq_Domain_Percentages': frequency_dict
        })

results_df = pd.DataFrame(results_list)

results_df.to_excel(output_excel_path, index=False)
print(f'Summary saved to {output_excel_path}')


## 05Plots_03_NetworkBurstParameters (Group-Condition_based plots) raw data 


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import sem
from matplotlib.backends.backend_pdf import PdfPages

file_path = NETWORK_BURST_XLSX
df = pd.read_excel(file_path)

output_dir = RESULTS_DIR / "05Plots_03_NetworkBurstParams_01"
os.makedirs(output_dir, exist_ok=True)

df = df[df['Sample_ID'] != 18574]

variables = ['Num_Peaks', 'Peak_Amplitudes', 'Peak_Frequencies', 'Peak_Widths', 'Synchronicity_Index']

for var in variables:
    df[var] = df[var].apply(lambda x: np.mean(eval(x)) if isinstance(x, str) and x.startswith('[') else x)

condition_order = ['Baseline', 'CBZ30uM', 'CBZ60uM', 'CBZ100uM', 'Washout']

df['Condition'] = pd.Categorical(df['Condition'], categories=condition_order, ordered=True)

grouped_data = df.groupby(['Group', 'Condition']).agg({
    'Num_Peaks': ['mean', 'sem', 'count'],
    'Peak_Amplitudes': ['mean', 'sem', 'count'],
    'Peak_Frequencies': ['mean', 'sem', 'count'],
    'Peak_Widths': ['mean', 'sem', 'count'],
    'Synchronicity_Index': ['mean', 'sem', 'count']
}).reset_index()

grouped_data.columns = ['Group', 'Condition'] + [f'{var}_{stat}' for var, stat in grouped_data.columns[2:]]

colors = {
    'AAV-Ctrl': '#1f77b4',
    'AAV-SMS': '#ff7f0e'
}

grouped_data['Condition'] = grouped_data['Condition'].astype(str)

for var in variables:
    fig, ax = plt.subplots(figsize=(8, 5))

    for group in df['Group'].unique():

        group_data = df[df['Group'] == group]

        sns.stripplot(data=group_data, x='Condition', y=var, color=colors[group], jitter=True, ax=ax)

        group_means = grouped_data[grouped_data['Group'] == group]
        ax.errorbar(group_means['Condition'], group_means[f'{var}_mean'],
                    yerr=group_means[f'{var}_sem'], fmt='-o',
                    color=colors[group], label=f'{group} Mean ± SEM',
                    linewidth=1,
                    markersize=8,
                    capsize=2,
                    elinewidth=0.5,
                    alpha=0.7)

    ax.set_title(f'{var} by Condition in Each Group')
    ax.set_xlabel('Condition')
    ax.set_ylabel(var)

    ax.legend(loc='upper left', bbox_to_anchor=(1, 1), frameon=False)

    plt.tight_layout(rect=[0, 0, 0.85, 1])

    pdf_path = os.path.join(output_dir, f"{var}_RawData_Plots.pdf")
    with PdfPages(pdf_path) as pdf:
        pdf.savefig(fig)

    plt.show()
    plt.close(fig)

print("All plots have been saved individually as PDFs.")


## Selected Conditions 05Plots_03_NetworkBurstParameters (Group-Condition_based plots) raw 


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import sem

file_path = NETWORK_BURST_XLSX
final_df = pd.read_excel(file_path)

output_dir = RESULTS_DIR / "Selected_05Plots_03_MEABasedNetBurst_01"
os.makedirs(output_dir, exist_ok=True)

output_stats_path = os.path.join(output_dir, 'Selected_MEABasedNetBurst_Summary.xlsx')

df = final_df[(final_df['Sample_ID'] != 18574) & (final_df['Condition'] != 'CBZ60uM') & (final_df['Condition'] != 'Washout')]

variables = ['Num_Peaks', 'Peak_Amplitudes', 'Peak_Frequencies', 'Peak_Widths', 'Synchronicity_Index']

for var in variables:
    df[var] = df[var].apply(lambda x: np.mean(eval(x)) if isinstance(x, str) and x.startswith('[') else x)

condition_order = ['Baseline', 'CBZ30uM', 'CBZ100uM']

df['Condition'] = pd.Categorical(df['Condition'], categories=condition_order, ordered=True)

excel_data = {}

group_colors = {
    'AAV-Ctrl': '#1f77b4',
    'AAV-SMS': '#ff7f0e'
}

for var in variables:
    fig, ax = plt.subplots(figsize=(4, 6))

    variable_data = []

    for group_idx, group in enumerate(df['Group'].unique()):

        group_data = df[df['Group'] == group]

        color = group_colors.get(group, 'black')

        boxplot_data = [
            group_data[group_data['Condition'] == condition].groupby('Sample_ID')[var].mean()
            for condition in condition_order
        ]

        for i, condition_data in enumerate(boxplot_data):

            boxplot_data[i] = condition_data.fillna(0)

        x_positions = np.arange(len(condition_order)) + 0.25 * group_idx
        box = plt.boxplot(
            boxplot_data,
            positions=x_positions,
            widths=0.25,
            patch_artist=True,
            boxprops=dict(facecolor=color, edgecolor='black', linewidth=0.7, alpha=0.7),
            medianprops=dict(color='black', linewidth=1),
            whiskerprops=dict(color='black', linewidth=0.7),
            capprops=dict(color='black', linewidth=0.7),
            flierprops=dict(marker='o', color=color, markersize=0, alpha=0.5),
            zorder=1
        )

        means = [np.mean(condition_data) for condition_data in boxplot_data]
        sems = [sem(condition_data) if len(condition_data) > 1 else 0 for condition_data in boxplot_data]

        ax.errorbar(
            np.arange(len(condition_order)) + 0.25 * group_idx,
            means,

            fmt='-o',
            color=color,
            label=f'{group} Mean',
            linewidth=1.5,
            markersize=8,
            markeredgecolor='black',
            alpha=0.9
        )

        for pos, sample_means in zip(x_positions, boxplot_data):
            jittered_x = pos + np.random.normal(0, 0.05, size=len(sample_means))
            ax.scatter(jittered_x, sample_means, color=color, edgecolor='black',
                       alpha=0.9, linewidth=0.5, zorder=3, s=40)

        for condition, sample_data in zip(condition_order, boxplot_data):
            for sample_id, mean_val in sample_data.items():

                if np.isnan(mean_val) or mean_val == 0:
                    mean_val = 0
                variable_data.append({
                    'Group': group,
                    'Condition': condition,
                    'Sample_ID': sample_id,
                    'Mean': mean_val
                })

    excel_data[var] = pd.DataFrame(variable_data)

    plt.title(f'{var} across Conditions',
              fontdict={'fontsize': 14, 'fontweight': 'bold', 'family': 'serif'})

    plt.xlabel('Condition', fontdict={'fontsize': 14, 'fontweight': 'medium', 'family': 'sans-serif'})

    plt.ylabel(var, fontdict={'fontsize': 14, 'fontweight': 'medium', 'family': 'sans-serif'})

    plt.legend(title="Groups", bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=14, title_fontsize=14)

    plt.xticks(ticks=np.arange(len(condition_order)), labels=condition_order, fontsize=14)

    plot_output_path = os.path.join(output_dir, f'{var}_Boxplot.pdf')
    plt.savefig(plot_output_path, format='pdf', bbox_inches='tight')
    plt.close()

with pd.ExcelWriter(output_stats_path, engine='openpyxl') as writer:
    for variable, data in excel_data.items():
        data.to_excel(writer, index=False, sheet_name=variable)

print(f"All plots have been saved as PDFs, and summary statistics have been exported to {output_stats_path}.")


## 05Plots_04_Normalized_NetworkBurstParameters (Group-Condition_based plots) Normalized 

- Saves Normalized variables together with original dataset in new excel file 05_NetBurstRH_01

- Consider that the original data lists in 03_NetBurstRH_01 has become values in 05_NetBurstRH_01 for each sample

- This data can be used for Prism because for all variables it has values (mean of the sample) --> 05_NetBurstRH_01

- Will be used from now on to plot normalized and original data


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import sem
from matplotlib.backends.backend_pdf import PdfPages

file_path = NETWORK_BURST_XLSX
df = pd.read_excel(file_path)

output_dir = RESULTS_DIR / "05Plots_04_NormNetworkBurstParams_01"
os.makedirs(output_dir, exist_ok=True)

df = df[df['Sample_ID'] != 18574]

variables = ['Num_Peaks', 'Peak_Amplitudes', 'Peak_Frequencies', 'Peak_Widths', 'Synchronicity_Index']

for var in variables:
    df[var] = df[var].apply(lambda x: np.mean(eval(x)) if isinstance(x, str) and x.startswith('[') else x)

for var in variables:
    df[var] = df[var].astype(float)

condition_order = ['Baseline', 'CBZ30uM', 'CBZ60uM', 'CBZ100uM', 'Washout']

df['Condition'] = pd.Categorical(df['Condition'], categories=condition_order, ordered=True)

for var in variables:
    df[f"{var}_Norm"] = np.nan

for sample_id in df['Sample_ID'].unique():
    sample_data = df[df['Sample_ID'] == sample_id]
    value_at_baseline = sample_data[sample_data['Condition'] == 'Baseline'][variables].replace(0, 0.001)

    for var in variables:
        if not value_at_baseline.empty:

            norm_values = (sample_data[var].values - value_at_baseline[var].values[0]) / value_at_baseline[var].values[0]

            df.loc[df['Sample_ID'] == sample_id, f"{var}_Norm"] = norm_values

output_excel_path = NETWORK_BURST_XLSX
df.to_excel(output_excel_path, index=False)
print(f"Combined original and normalized data saved to: {output_excel_path}")

grouped_data = df.groupby(['Group', 'Condition']).agg({
    'Num_Peaks_Norm': ['mean', 'sem', 'count'],
    'Peak_Amplitudes_Norm': ['mean', 'sem', 'count'],
    'Peak_Frequencies_Norm': ['mean', 'sem', 'count'],
    'Peak_Widths_Norm': ['mean', 'sem', 'count'],
    'Synchronicity_Index_Norm': ['mean', 'sem', 'count']
}).reset_index()

grouped_data.columns = ['Group', 'Condition'] + [f'{var}_{stat}' for var, stat in grouped_data.columns[2:]]

colors = {
    'AAV-Ctrl': '#1f77b4',
    'AAV-SMS': '#ff7f0e'
}

for var in [f"{v}_Norm" for v in variables]:
    fig, ax = plt.subplots(figsize=(8, 5))

    for group in df['Group'].unique():
        group_data = df[df['Group'] == group]

        sns.stripplot(data=group_data, x='Condition', y=var, color=colors[group], jitter=True, ax=ax)

        group_means = grouped_data[grouped_data['Group'] == group]
        ax.errorbar(group_means['Condition'], group_means[f'{var}_mean'],
                    yerr=group_means[f'{var}_sem'], fmt='-o',
                    color=colors[group], label=f'{group} Mean ± SEM',
                    linewidth=1,
                    markersize=8,
                    capsize=2,
                    elinewidth=0.5,
                    alpha=0.7)

    ax.set_title(f'{var} by Condition in Each Group (Normalized)')
    ax.set_xlabel('Condition')
    ax.set_ylabel(var)
    ax.legend(loc='upper left', bbox_to_anchor=(1, 1), frameon=False)

    plt.tight_layout(rect=[0, 0, 0.85, 1])

    pdf_path = os.path.join(output_dir, f"{var}_Plot.pdf")
    with PdfPages(pdf_path) as pdf:
        pdf.savefig(fig)

    plt.show()
    plt.close(fig)

print("All normalized plots have been saved individually as PDFs.")


## Normalized _ Selected Conditions 05Plots_03_NetworkBurstParameters (Group-Condition_based plots) Normalized data CBZ01


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import sem

file_path = NETWORK_BURST_XLSX
final_df = pd.read_excel(file_path)

output_dir = RESULTS_DIR / "Selected_05Plots_03_MEABasedNetBurst_01"
os.makedirs(output_dir, exist_ok=True)

output_stats_path = os.path.join(output_dir, 'Selected_NormMEABasedNetBurst_Summary.xlsx')

df = final_df[(final_df['Sample_ID'] != 18574) & (final_df['Condition'] != 'CBZ60uM') & (final_df['Condition'] != 'Washout')]

variables = ['Num_Peaks', 'Peak_Amplitudes', 'Peak_Frequencies', 'Peak_Widths', 'Synchronicity_Index']

for var in variables:
    df[var] = df[var].apply(lambda x: np.mean(eval(x)) if isinstance(x, str) and x.startswith('[') else x)

condition_order = ['Baseline', 'CBZ30uM', 'CBZ100uM']

df['Condition'] = pd.Categorical(df['Condition'], categories=condition_order, ordered=True)

excel_data = {}

group_colors = {
    'AAV-Ctrl': '#1f77b4',
    'AAV-SMS': '#ff7f0e'
}

for var in variables:
    fig, ax = plt.subplots(figsize=(4, 6))

    variable_data = []

    for group_idx, group in enumerate(df['Group'].unique()):

        group_data = df[df['Group'] == group]

        color = group_colors.get(group, 'black')

        boxplot_data = [
            group_data[group_data['Condition'] == condition].groupby('Sample_ID')[var].mean()
            for condition in condition_order
        ]

        for i, condition_data in enumerate(boxplot_data):
            if condition_data.empty:
                boxplot_data[i] = pd.Series([0] * len(group_data['Sample_ID'].unique()), index=group_data['Sample_ID'].unique())
            else:

                boxplot_data[i] = boxplot_data[i].fillna(0)

        baseline_means = boxplot_data[0]

        for i in range(0, len(boxplot_data)):
            boxplot_data[i] = ((boxplot_data[i] - baseline_means) / baseline_means) * 100

            boxplot_data[i] = boxplot_data[i].fillna(0)

        print(boxplot_data)

        x_positions = np.arange(len(condition_order)) + 0.25 * group_idx
        box = plt.boxplot(
            boxplot_data,
            positions=x_positions,
            widths=0.25,
            patch_artist=True,
            boxprops=dict(facecolor=color, edgecolor='black', linewidth=0.7, alpha=0.7),
            medianprops=dict(color='black', linewidth=1),
            whiskerprops=dict(color='black', linewidth=0.7),
            capprops=dict(color='black', linewidth=0.7),
            flierprops=dict(marker='o', color=color, markersize=0, alpha=0.5),
            zorder=1
        )

        means = [np.mean(condition_data) for condition_data in boxplot_data]
        sems = [sem(condition_data) if len(condition_data) > 1 else 0 for condition_data in boxplot_data]

        ax.errorbar(
            np.arange(len(condition_order)) + 0.25 * group_idx,
            means,

            fmt='-o',
            color=color,
            label=f'{group} Mean',
            linewidth=1.5,
            markersize=8,
            markeredgecolor='black',
            alpha=0.9
        )

        for pos, sample_means in zip(x_positions, boxplot_data):
            jittered_x = pos + np.random.normal(0, 0.05, size=len(sample_means))
            ax.scatter(jittered_x, sample_means, color=color, edgecolor='black',
                       alpha=0.9, linewidth=0.5, zorder=3, s=40)

        for condition, sample_data in zip(condition_order, boxplot_data):
            for sample_id, mean_val in sample_data.items():

                mean_val = 0 if np.isnan(mean_val) else mean_val
                variable_data.append({
                    'Group': group,
                    'Condition': condition,
                    'Sample_ID': sample_id,
                    'Mean': mean_val
                })

    excel_data[var] = pd.DataFrame(variable_data)

    plt.title(f'{var} across Conditions (Normalized)',
              fontdict={'fontsize': 14, 'fontweight': 'bold', 'family': 'serif'})

    plt.xlabel('Condition', fontdict={'fontsize': 14, 'fontweight': 'medium', 'family': 'sans-serif'})

    plt.ylabel(f'Normalized {var}', fontdict={'fontsize': 14, 'fontweight': 'medium', 'family': 'sans-serif'})

    plt.legend(title="Groups", bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=14, title_fontsize=14)

    plt.xticks(ticks=np.arange(len(condition_order)), labels=condition_order, fontsize=14)

    plot_output_path = os.path.join(output_dir, f'{var}_Normalized_Boxplot.pdf')
    plt.savefig(plot_output_path, format='pdf', bbox_inches='tight')
    plt.close()

with pd.ExcelWriter(output_stats_path, engine='openpyxl') as writer:
    for variable, data in excel_data.items():
        data.to_excel(writer, index=False, sheet_name=variable)

print(f"All plots have been saved as PDFs, and summary statistics have been exported to {output_stats_path}.")


## 05Plots_05_Conditionbased_FreqDomains

- One plot for each Condition in two groups

- uses the already calculated percentages of bins with specific freq in excel file


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

plt.rcParams['text.usetex'] = False
plt.rcParams['font.family'] = 'Arial'

output_excel_path = NETWORK_BURST_XLSX
results_df = pd.read_excel(output_excel_path)

output_dir = RESULTS_DIR / "05Plots_05_FreqDomainsConditions"
os.makedirs(output_dir, exist_ok=True)

freq_data = []

for idx, row in results_df.iterrows():
    group = row['Group']
    condition = row['Condition']
    freq_domain_dict = eval(row['Freq_Domain_Percentages'])
    freq_data.append({'Group': group, 'Condition': condition, **{int(k.strip('Hz')): float(v.strip('%')) for k, v in freq_domain_dict.items()}})

freq_df = pd.DataFrame(freq_data)

summary_df = freq_df.groupby(['Group', 'Condition']).agg(['mean', 'std']).reset_index()

y_min, y_max = 0.01, 100
x_min, x_max = 0.0, 50

condition_order = ['Baseline', 'CBZ30uM', 'CBZ60uM', 'CBZ100uM', 'Washout']

freq_df['Condition'] = pd.Categorical(freq_df['Condition'], categories=condition_order, ordered=True)

plt.figure(figsize=(8, 6))

for group in summary_df['Group'].unique():
    group_df = summary_df[summary_df['Group'] == group]

    freqs = range(101)

    for condition in condition_order:
        condition_df = group_df[group_df['Condition'] == condition]

        mean_percentages = condition_df.iloc[:, condition_df.columns.get_level_values(1) == 'mean'].values.flatten()
        std_percentages = condition_df.iloc[:, condition_df.columns.get_level_values(1) == 'std'].values.flatten()

        mean_line, = plt.plot(freqs, mean_percentages, label=f'{group} - {condition} Mean')

        yerr = np.array([np.zeros_like(std_percentages), std_percentages])

        plt.errorbar(
            freqs, mean_percentages, yerr=yerr, fmt='o',
            color=mean_line.get_color(), alpha=0.5, elinewidth=0.8, capsize=2
        )

plt.yscale('log')
plt.ylim(y_min, y_max)
plt.xlim(x_min, x_max)
plt.title('Frequency Domain Percentages for All Conditions and Groups')
plt.xlabel('Frequency (Hz)')
plt.ylabel('Percentage (%) (Log Scale)')
plt.legend()

pdf_file_path = os.path.join(output_dir, 'Freq_Bin_Condition_based.pdf')
plt.savefig(pdf_file_path, format='pdf')

plt.show()
plt.close()

for group in summary_df['Group'].unique():
    group_df = summary_df[summary_df['Group'] == group]

    plt.figure(figsize=(8, 6))

    for condition in condition_order:
        condition_df = group_df[group_df['Condition'] == condition]

        mean_percentages = condition_df.iloc[:, condition_df.columns.get_level_values(1) == 'mean'].values.flatten()
        std_percentages = condition_df.iloc[:, condition_df.columns.get_level_values(1) == 'std'].values.flatten()

        mean_line, = plt.plot(freqs, mean_percentages, label=f'{condition} Mean')

        yerr = np.array([np.zeros_like(std_percentages), std_percentages])

        plt.errorbar(
            freqs, mean_percentages, yerr=yerr, fmt='o',
            color=mean_line.get_color(), alpha=0.5, elinewidth=0.8, capsize=2
        )

    plt.yscale('log')
    plt.ylim(y_min, y_max)
    plt.xlim(x_min, x_max)
    plt.title(f'Frequency Domain Percentages for {group}')
    plt.xlabel('Frequency (Hz)')
    plt.ylabel('Percentage (%) (Log Scale)')
    plt.legend()

    pdf_file_path_group = os.path.join(output_dir, f'Freq_Bin_Condition_based_{group}.pdf')
    plt.savefig(pdf_file_path_group, format='pdf')

    plt.show()
    plt.close()


## Selected Conditions ___ 05Plots_05_Conditionbased_FreqDomains

- One plot for each Condition in two groups

- uses the already calculated percentages of bins with specific freq in excel file


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

plt.rcParams['text.usetex'] = False
plt.rcParams['font.family'] = 'Arial'

output_excel_path = NETWORK_BURST_XLSX
results_df = pd.read_excel(output_excel_path)

output_dir = RESULTS_DIR / "Selected_05Plots_05_FreqDomainsConditions"
os.makedirs(output_dir, exist_ok=True)

freq_data = []

for idx, row in results_df.iterrows():
    group = row['Group']
    condition = row['Condition']
    freq_domain_dict = eval(row['Freq_Domain_Percentages'])
    freq_data.append({'Group': group, 'Condition': condition, **{int(k.strip('Hz')): float(v.strip('%')) for k, v in freq_domain_dict.items()}})

freq_df = pd.DataFrame(freq_data)

summary_df = freq_df.groupby(['Group', 'Condition']).agg(['mean', 'std']).reset_index()

y_min, y_max = 0.01, 100
x_min, x_max = 0.0, 40

condition_map = {'00': 'Baseline', '01':'CBZ30uM', '02': 'CBZ100uM'}

condition_order = [condition_map['00'], condition_map['01'], condition_map['02']]

freq_df['Condition'] = pd.Categorical(freq_df['Condition'], categories=condition_order, ordered=True)

condition_colors = {}
for condition in condition_order:
    if condition in ['Baseline']:
        condition_colors[condition] = '#4F4F4F'
    else:
        condition_colors[condition] = None

condition_intensity = {
    'AAV-Ctrl': {
        'CBZ30uM': '#4682B4',
        'CBZ100uM': '#00008B'
    },
    'AAV-SMS': {
        'CBZ30uM': '#CD5C5C',
        'CBZ100uM': '#FF4500'
    }
}

plt.figure(figsize=(4, 6))

for group in summary_df['Group'].unique():
    group_df = summary_df[summary_df['Group'] == group]

    freqs = range(101)

    for condition in condition_order:
        condition_df = group_df[group_df['Condition'] == condition]

        mean_percentages = condition_df.iloc[:, condition_df.columns.get_level_values(1) == 'mean'].values.flatten()
        std_percentages = condition_df.iloc[:, condition_df.columns.get_level_values(1) == 'std'].values.flatten()

        if group == 'AAV-Ctrl':
            if condition == 'CBZ30uM':
                color = condition_intensity['AAV-Ctrl']['CBZ30uM']
            elif condition == 'CBZ100uM':
                color = condition_intensity['AAV-Ctrl']['CBZ100uM']
            else:
                color = '#4F4F4F'
        elif group == 'AAV-SMS':
            if condition == 'CBZ30uM':
                color = condition_intensity['AAV-SMS']['CBZ30uM']
            elif condition == 'CBZ100uM':
                color = condition_intensity['AAV-SMS']['CBZ100uM']
            else:
                color = '#4F4F4F'

        mean_line, = plt.plot(freqs, mean_percentages, label=f'{group} - {condition} Mean', color=color, linewidth=0.8)

        yerr = np.array([np.zeros_like(std_percentages), std_percentages])

        plt.errorbar(
            freqs, mean_percentages, yerr=yerr, fmt='o',
            color=mean_line.get_color(), alpha=0.5, elinewidth=0.6, capsize=2, markersize=4
        )

plt.yscale('log')
plt.ylim(y_min, y_max)
plt.xlim(x_min, x_max)

plt.title('Frequency Domain Percentages for All Conditions and Groups', fontsize=14)

plt.xlabel('Frequency (Hz)', fontsize=14)
plt.ylabel('Percentage (%) (Log Scale)', fontsize=14)

plt.legend(fontsize=14, loc='upper left', bbox_to_anchor=(1.05, 1))

plt.xticks(fontsize=14)
plt.yticks(fontsize=14)

pdf_file_path = os.path.join(output_dir, 'ConditionsFreqDomain.pdf')
plt.savefig(pdf_file_path, format='pdf', bbox_inches='tight')

plt.show()
plt.close()

for group in summary_df['Group'].unique():
    group_df = summary_df[summary_df['Group'] == group]

    plt.figure(figsize=(4, 6))

    for condition in condition_order:
        condition_df = group_df[group_df['Condition'] == condition]

        mean_percentages = condition_df.iloc[:, condition_df.columns.get_level_values(1) == 'mean'].values.flatten()
        std_percentages = condition_df.iloc[:, condition_df.columns.get_level_values(1) == 'std'].values.flatten()

        if group == 'AAV-Ctrl':
            if condition == 'CBZ30uM':
                color = condition_intensity['AAV-Ctrl']['CBZ30uM']
            elif condition == 'CBZ100uM':
                color = condition_intensity['AAV-Ctrl']['CBZ100uM']
            else:
                color = '#4F4F4F'
        elif group == 'AAV-SMS':
            if condition == 'CBZ30uM':
                color = condition_intensity['AAV-SMS']['CBZ30uM']
            elif condition == 'CBZ100uM':
                color = condition_intensity['AAV-SMS']['CBZ100uM']
            else:
                color = '#4F4F4F'

        mean_line, = plt.plot(freqs, mean_percentages, label=f'{condition} Mean', color=color, linewidth=0.8)

        yerr = np.array([np.zeros_like(std_percentages), std_percentages])

        plt.errorbar(
            freqs, mean_percentages, yerr=yerr, fmt='o',
            color=mean_line.get_color(), alpha=0.5, elinewidth=0.6, capsize=2, markersize=4
        )

    plt.yscale('log')
    plt.ylim(y_min, y_max)
    plt.xlim(x_min, x_max)
    plt.title(f'Frequency Domain Percentages for {group}')

    plt.xlabel('Frequency (Hz)', fontsize=14)
    plt.ylabel('Percentage (%) (Log Scale)', fontsize=14)

    plt.legend(fontsize=14, loc='upper left', bbox_to_anchor=(1.05, 1))

    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)

    pdf_file_path_group = os.path.join(output_dir, f'{group}_ConditionsFreqDomain.pdf')
    plt.savefig(pdf_file_path_group, format='pdf', bbox_inches='tight')

    plt.show()
    plt.close()


## 20261006  to extract excel data of plots for manuscript


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

plt.rcParams['text.usetex'] = False
plt.rcParams['font.family'] = 'Arial'

output_excel_path = NETWORK_BURST_XLSX
output_dir = RESULTS_DIR / "Selected_05Plots_05_FreqDomainsConditions_2026ForKunihiko"
os.makedirs(output_dir, exist_ok=True)

plot_data_excel = os.path.join(
    output_dir,
    'ConditionsFreqDomain_PlotData_0to40Hz.xlsx'
)

results_df = pd.read_excel(output_excel_path)

freq_data = []

for idx, row in results_df.iterrows():

    group = row['Group']
    condition = row['Condition']

    freq_domain_dict = eval(row['Freq_Domain_Percentages'])

    parsed_data = {
        int(k.strip('Hz')): float(v.strip('%'))
        for k, v in freq_domain_dict.items()
    }

    freq_data.append({
        'Group': group,
        'Condition': condition,
        **parsed_data
    })

freq_df = pd.DataFrame(freq_data)

condition_map = {
    '00': 'Baseline',
    '01': 'CBZ30uM',
    '02': 'CBZ100uM'
}

condition_order = [
    condition_map['00'],
    condition_map['01'],
    condition_map['02']
]

freq_df['Condition'] = pd.Categorical(
    freq_df['Condition'],
    categories=condition_order,
    ordered=True
)

summary_df = (
    freq_df
    .groupby(
        ['Group', 'Condition'],
        observed=True
    )
    .agg(['mean', 'std'])
    .reset_index()
)

freqs = np.arange(0, 101)

export_freqs = np.arange(0, 41)

y_min = 0.01
y_max = 100

x_min = 0
x_max = 40

condition_intensity = {

    'AAV-Ctrl': {
        'CBZ30uM': '#4682B4',
        'CBZ100uM': '#00008B'
    },

    'AAV-SMS': {
        'CBZ30uM': '#CD5C5C',
        'CBZ100uM': '#FF4500'
    }
}

def get_color(group, condition):

    if condition == 'Baseline':
        return '#4F4F4F'

    if group in condition_intensity:

        if condition in condition_intensity[group]:
            return condition_intensity[group][condition]

    return None

def get_plot_values(group, condition):

    condition_df = summary_df[
        (summary_df['Group'] == group) &
        (summary_df['Condition'] == condition)
    ]

    if condition_df.empty:
        return None, None

    means = []
    stds = []

    for freq in freqs:

        if (freq, 'mean') in condition_df.columns:
            mean_value = condition_df[(freq, 'mean')].iloc[0]
        else:
            mean_value = np.nan

        if (freq, 'std') in condition_df.columns:
            std_value = condition_df[(freq, 'std')].iloc[0]
        else:
            std_value = np.nan

        means.append(mean_value)
        stds.append(std_value)

    return (
        np.array(means, dtype=float),
        np.array(stds, dtype=float)
    )

all_plot_data = []

group_plot_data = {
    group: []
    for group in summary_df['Group'].unique()
}

for group in summary_df['Group'].unique():

    for condition in condition_order:

        mean_percentages, std_percentages = get_plot_values(
            group,
            condition
        )

        if mean_percentages is None:
            continue

        export_mean = mean_percentages[:41]
        export_std = std_percentages[:41]

        temp_df = pd.DataFrame({

            'Group': [group] * len(export_freqs),

            'Condition': [condition] * len(export_freqs),

            'Frequency_Hz': export_freqs,

            'Mean_Percentage': export_mean,

            'SD_Percentage': export_std,

            'Mean_plus_SD': export_mean + export_std
        })

        all_plot_data.append(temp_df)

        group_plot_data[group].append(
            temp_df.copy()
        )

all_plot_data_df = pd.concat(
    all_plot_data,
    ignore_index=True
)

with pd.ExcelWriter(
    plot_data_excel,
    engine='openpyxl'
) as writer:

    all_plot_data_df.to_excel(
        writer,
        sheet_name='All_Groups',
        index=False
    )

    for group, dfs in group_plot_data.items():

        if len(dfs) == 0:
            continue

        group_export_df = pd.concat(
            dfs,
            ignore_index=True
        )

        sheet_name = str(group)[:31]

        group_export_df.to_excel(
            writer,
            sheet_name=sheet_name,
            index=False
        )

print('Excel plot data exported successfully.')
print('Only frequencies 0–40 Hz were exported.')
print(plot_data_excel)

plt.figure(figsize=(4, 6))

for group in summary_df['Group'].unique():

    for condition in condition_order:

        mean_percentages, std_percentages = get_plot_values(
            group,
            condition
        )

        if mean_percentages is None:
            continue

        color = get_color(
            group,
            condition
        )

        mean_line, = plt.plot(
            freqs,
            mean_percentages,
            label=f'{group} - {condition} Mean',
            color=color,
            linewidth=0.8
        )

        yerr = np.array([
            np.zeros_like(std_percentages),
            std_percentages
        ])

        plt.errorbar(
            freqs,
            mean_percentages,
            yerr=yerr,
            fmt='o',
            color=mean_line.get_color(),
            alpha=0.5,
            elinewidth=0.6,
            capsize=2,
            markersize=4
        )

plt.yscale('log')

plt.ylim(
    y_min,
    y_max
)

plt.xlim(
    x_min,
    x_max
)

plt.title(
    'Frequency Domain Percentages for All Conditions and Groups',
    fontsize=14
)

plt.xlabel(
    'Frequency (Hz)',
    fontsize=14
)

plt.ylabel(
    'Percentage (%) (Log Scale)',
    fontsize=14
)

plt.legend(
    fontsize=14,
    loc='upper left',
    bbox_to_anchor=(1.05, 1)
)

plt.xticks(
    fontsize=14
)

plt.yticks(
    fontsize=14
)

pdf_file_path = os.path.join(
    output_dir,
    'ConditionsFreqDomain.pdf'
)

plt.savefig(
    pdf_file_path,
    format='pdf',
    bbox_inches='tight'
)

plt.show()

plt.close()

for group in summary_df['Group'].unique():

    plt.figure(figsize=(4, 6))

    for condition in condition_order:

        mean_percentages, std_percentages = get_plot_values(
            group,
            condition
        )

        if mean_percentages is None:
            continue

        color = get_color(
            group,
            condition
        )

        mean_line, = plt.plot(
            freqs,
            mean_percentages,
            label=f'{condition} Mean',
            color=color,
            linewidth=0.8
        )

        yerr = np.array([
            np.zeros_like(std_percentages),
            std_percentages
        ])

        plt.errorbar(
            freqs,
            mean_percentages,
            yerr=yerr,
            fmt='o',
            color=mean_line.get_color(),
            alpha=0.5,
            elinewidth=0.6,
            capsize=2,
            markersize=4
        )

    plt.yscale('log')

    plt.ylim(
        y_min,
        y_max
    )

    plt.xlim(
        x_min,
        x_max
    )

    plt.title(
        f'Frequency Domain Percentages for {group}'
    )

    plt.xlabel(
        'Frequency (Hz)',
        fontsize=14
    )

    plt.ylabel(
        'Percentage (%) (Log Scale)',
        fontsize=14
    )

    plt.legend(
        fontsize=14,
        loc='upper left',
        bbox_to_anchor=(1.05, 1)
    )

    plt.xticks(
        fontsize=14
    )

    plt.yticks(
        fontsize=14
    )

    pdf_file_path_group = os.path.join(
        output_dir,
        f'{group}_ConditionsFreqDomain.pdf'
    )

    plt.savefig(
        pdf_file_path_group,
        format='pdf',
        bbox_inches='tight'
    )

    plt.show()

    plt.close()

print()
print('Finished successfully.')
print()
print('Plots saved in:')
print(output_dir)
print()
print('Excel data saved as:')
print(plot_data_excel)


## 05_07_Normalize the Freq_Domain_Percentages --> Freq_Domain_Percentages_Norm --- Excel output

- The new excel file contain both normalized data and original data

- Normalized dictionary is saved in column Freq_Domain_Percentages_Norm


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ast

output_excel_path = NETWORK_BURST_XLSX
results_df = pd.read_excel(output_excel_path)

freq_data = []
for idx, row in results_df.iterrows():
    if row.get('File Name') != '20240812_18389_01':
        group = row['Group']
        condition = row['Condition']
        sample_id = int(row['Sample_ID'])
        freq_domain_dict = eval(row['Freq_Domain_Percentages'])
        freq_data.append({
            'Group': group, 'Condition': condition, 'Sample_ID': sample_id,
            **{int(k.strip('Hz')): float(v.strip('%')) for k, v in freq_domain_dict.items()}
        })

freq_df = pd.DataFrame(freq_data)

normalized_data = []

for sample_id in freq_df['Sample_ID'].unique():
    sample_df = freq_df[freq_df['Sample_ID'] == sample_id]
    baseline_row = sample_df[sample_df['Condition'] == 'Baseline']

    if not baseline_row.empty:
        baseline_values = baseline_row.iloc[0, 3:].to_dict()

        for _, row in sample_df.iterrows():
            condition = row['Condition']
            group = row['Group']
            normalized_row = {'Group': group, 'Condition': condition, 'Sample_ID': sample_id}
            freq_norm_dict = {}

            for key, value in row[3:].to_dict().items():

                norm_value = ((value - baseline_values[key]) / baseline_values[key]) * 100 if baseline_values[key] != 0 else ((value - baseline_values[key]) / 0.001) * 100
                freq_norm_dict[f'{key}Hz'] = f'{norm_value:.2f}%'

            normalized_row['Freq_Domain_Percentages_Norm'] = freq_norm_dict
            normalized_data.append(normalized_row)

normalized_df = pd.DataFrame(normalized_data)

merged_df = pd.merge(results_df, normalized_df[['Group', 'Condition', 'Sample_ID', 'Freq_Domain_Percentages_Norm']],
                     on=['Group', 'Condition', 'Sample_ID'], how='left')

output_excel_path_normalized = NORMALIZED_FREQUENCY_XLSX
merged_df.to_excel(output_excel_path_normalized, index=False)
print(f'Normalized data together with original data were saved to {output_excel_path_normalized}')


## Selected Conditions _ 05_07_Normalize the Freq_Domain_Percentages --> Freq_Domain_Percentages_Norm --- Excel output

- The new excel file contain both normalized data and original data

- Normalized dictionary is saved in column Freq_Domain_Percentages_Norm


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ast

output_excel_path = NETWORK_BURST_XLSX
results_df = pd.read_excel(output_excel_path)

freq_data = []
for idx, row in results_df.iterrows():
    if row.get('File Name') != '20240812_18389_01':
        group = row['Group']
        condition = row['Condition']
        sample_id = int(row['Sample_ID'])
        freq_domain_dict = eval(row['Freq_Domain_Percentages'])
        freq_data.append({
            'Group': group, 'Condition': condition, 'Sample_ID': sample_id,
            **{int(k.strip('Hz')): float(v.strip('%')) for k, v in freq_domain_dict.items()}
        })

freq_df = pd.DataFrame(freq_data)

normalized_data = []

for sample_id in freq_df['Sample_ID'].unique():
    sample_df = freq_df[freq_df['Sample_ID'] == sample_id]
    baseline_row = sample_df[sample_df['Condition'] == 'Baseline']

    if not baseline_row.empty:
        baseline_values = baseline_row.iloc[0, 3:].to_dict()

        for _, row in sample_df.iterrows():
            condition = row['Condition']
            group = row['Group']
            normalized_row = {'Group': group, 'Condition': condition, 'Sample_ID': sample_id}
            freq_norm_dict = {}

            for key, value in row[3:].to_dict().items():

                norm_value = ((value - baseline_values[key]) / baseline_values[key]) * 100 if baseline_values[key] != 0 else ((value - baseline_values[key]) / 0.001) * 100
                freq_norm_dict[f'{key}Hz'] = f'{norm_value:.2f}%'

            normalized_row['Freq_Domain_Percentages_Norm'] = freq_norm_dict
            normalized_data.append(normalized_row)

normalized_df = pd.DataFrame(normalized_data)

merged_df = pd.merge(results_df, normalized_df[['Group', 'Condition', 'Sample_ID', 'Freq_Domain_Percentages_Norm']],
                     on=['Group', 'Condition', 'Sample_ID'], how='left')

output_excel_path_normalized = NORMALIZED_FREQUENCY_XLSX
merged_df.to_excel(output_excel_path_normalized, index=False)
print(f'Normalized data together with original data were saved to {output_excel_path_normalized}')


## 05Plots_07_NormalizedFreqDomains Normalized Condition_based plots

- Uses the new excel file contain both normalized data and original data

- uses the already calculated normalized percentages of bins with specific freq in excel file


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

output_excel_path = NORMALIZED_FREQUENCY_XLSX
results_df = pd.read_excel(output_excel_path)

save_dir = RESULTS_DIR / "05Plots_07_NormFreqDomainsConditions"
os.makedirs(save_dir, exist_ok=True)

freq_data = []
for idx, row in results_df.iterrows():
    if row.get('File Name') != '20240812_18389_01':
        group = row['Group']
        condition = row['Condition']
        sample_id = int(row['Sample_ID'])

        freq_domain_dict = eval(row['Freq_Domain_Percentages_Norm'])
        freq_data.append({
            'Group': group, 'Condition': condition, 'Sample_ID': sample_id,
            **{int(k.strip('Hz')): float(v.strip('%')) for k, v in freq_domain_dict.items()}
        })

normalized_df = pd.DataFrame(freq_data)

normalized_df_filtered = normalized_df[normalized_df['Sample_ID'] != 18574].copy()

frequency_columns = normalized_df_filtered.columns[3:]

condition_order = ['Baseline', 'CBZ30uM', 'CBZ60uM', 'CBZ100uM', 'Washout']

normalized_df_filtered['Condition'] = pd.Categorical(normalized_df_filtered['Condition'], categories=condition_order, ordered=True)

def sem(x):
    return np.std(x, ddof=1) / np.sqrt(len(x))

unique_conditions = condition_order
groups = normalized_df_filtered['Group'].unique()

for condition in unique_conditions:
    fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(6, 6), gridspec_kw={'height_ratios': [1, 3]})

    for group in groups:

        group_condition_data = normalized_df_filtered[(normalized_df_filtered['Group'] == group) &
                                                      (normalized_df_filtered['Condition'] == condition)]

        if not group_condition_data.empty:

            mean_percentages = group_condition_data[frequency_columns].mean()
            sem_percentages = group_condition_data[frequency_columns].apply(sem)
            freqs = np.arange(len(frequency_columns))

            plot_color = 'C1' if group == 'AAV-SMS' else 'C0'

            ax1.plot(freqs, mean_percentages, label=f'{group} Mean', color=plot_color)
            ax1.fill_between(freqs, mean_percentages - sem_percentages, mean_percentages + sem_percentages,
                             color=plot_color, alpha=0.2)
            ax2.plot(freqs, mean_percentages, label=f'{group} Mean', color=plot_color)
            ax2.fill_between(freqs, mean_percentages - sem_percentages, mean_percentages + sem_percentages,
                             color=plot_color, alpha=0.2)

            for i, freq in enumerate(frequency_columns):
                freq_values = group_condition_data[freq].values
                ax1.scatter([freqs[i]] * len(freq_values), freq_values, color=plot_color, alpha=0.3, s=3)
                ax2.scatter([freqs[i]] * len(freq_values), freq_values, color=plot_color, alpha=0.3, s=3)

    ax1.set_ylim(200, 400)
    ax2.set_ylim(-100, 200)
    plt.xlim(0, 40)

    ax1.spines['bottom'].set_visible(False)
    ax2.spines['top'].set_visible(False)
    ax1.xaxis.tick_top()
    ax1.tick_params(labeltop=False)
    ax2.xaxis.tick_bottom()

    d = 0.01
    kwargs = dict(transform=ax1.transAxes, color='k', clip_on=False)
    ax1.plot((-d, +d), (-d, +d), **kwargs)
    ax1.plot((1 - d, 1 + d), (-d, +d), **kwargs)

    kwargs.update(transform=ax2.transAxes)
    ax2.plot((-d, +d), (1 - d, 1 + d), **kwargs)
    ax2.plot((1 - d, 1 + d), (1 - d, 1 + d), **kwargs)

    fig.suptitle(f'Averaged Normalized Frequency Domain Percentages for Condition {condition}')
    ax2.set_xlabel('Frequency (Hz)')
    ax1.set_ylabel('Normalized Percentage (%)')
    ax2.set_ylabel('Normalized Percentage (%)')

    ax2.legend(loc='upper left', bbox_to_anchor=(1, 1))
    plt.tight_layout()
    plt.subplots_adjust(hspace=0.00)

    save_path = os.path.join(save_dir, f'Freq_Bin_condition_based_{condition}_Norm.pdf')
    plt.savefig(save_path, format='pdf')
    plt.close(fig)


## 20261006--> Extraction of excel data for data for manuscript


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import ast

output_excel_path = NORMALIZED_FREQUENCY_XLSX
results_df = pd.read_excel(output_excel_path)

save_dir = RESULTS_DIR / "05Plots_07_NormFreqDomainsConditions_2026_----"
os.makedirs(save_dir, exist_ok=True)

plot_data_excel = os.path.join(
    save_dir,
    'Freq_Bin_condition_based_Norm_PlotData.xlsx'
)

freq_data = []

for idx, row in results_df.iterrows():

    if row.get('File Name') != '20240812_18389_01':

        group = row['Group']
        condition = row['Condition']
        sample_id = int(row['Sample_ID'])

        freq_domain_dict = ast.literal_eval(
            row['Freq_Domain_Percentages_Norm']
        )

        parsed_values = {
            int(k.strip('Hz')): float(v.strip('%'))
            for k, v in freq_domain_dict.items()
        }

        freq_data.append({
            'Group': group,
            'Condition': condition,
            'Sample_ID': sample_id,
            **parsed_values
        })

normalized_df = pd.DataFrame(freq_data)

normalized_df_filtered = normalized_df[
    normalized_df['Sample_ID'] != 18574
].copy()

condition_order = [
    'Baseline',
    'CBZ30uM',
    'CBZ60uM',
    'CBZ100uM',
    'Washout'
]

normalized_df_filtered['Condition'] = pd.Categorical(
    normalized_df_filtered['Condition'],
    categories=condition_order,
    ordered=True
)

plot_frequencies = list(range(0, 41))

frequency_columns = [
    freq for freq in plot_frequencies
    if freq in normalized_df_filtered.columns
]

def sem(x):

    x = pd.Series(x).dropna()

    if len(x) <= 1:
        return np.nan

    return np.std(x, ddof=1) / np.sqrt(len(x))

groups = normalized_df_filtered['Group'].dropna().unique()

with pd.ExcelWriter(
    plot_data_excel,
    engine='openpyxl'
) as writer:

    for condition in condition_order:

        fig, (ax1, ax2) = plt.subplots(
            2,
            1,
            sharex=True,
            figsize=(6, 6),
            gridspec_kw={'height_ratios': [1, 3]}
        )

        condition_export_data = []

        for group in groups:

            group_condition_data = normalized_df_filtered[
                (normalized_df_filtered['Group'] == group) &
                (normalized_df_filtered['Condition'] == condition)
            ]

            if group_condition_data.empty:
                continue

            mean_percentages = (
                group_condition_data[frequency_columns]
                .mean()
            )

            sem_percentages = (
                group_condition_data[frequency_columns]
                .apply(sem)
            )

            freqs = np.array(frequency_columns)

            plot_color = 'C1' if group == 'AAV-SMS' else 'C0'

            ax1.plot(
                freqs,
                mean_percentages.values,
                label=f'{group} Mean',
                color=plot_color
            )

            ax1.fill_between(
                freqs,
                (mean_percentages - sem_percentages).values,
                (mean_percentages + sem_percentages).values,
                color=plot_color,
                alpha=0.2
            )

            ax2.plot(
                freqs,
                mean_percentages.values,
                label=f'{group} Mean',
                color=plot_color
            )

            ax2.fill_between(
                freqs,
                (mean_percentages - sem_percentages).values,
                (mean_percentages + sem_percentages).values,
                color=plot_color,
                alpha=0.2
            )

            for freq in frequency_columns:

                freq_values = (
                    group_condition_data[freq]
                    .values
                )

                ax1.scatter(
                    [freq] * len(freq_values),
                    freq_values,
                    color=plot_color,
                    alpha=0.3,
                    s=3
                )

                ax2.scatter(
                    [freq] * len(freq_values),
                    freq_values,
                    color=plot_color,
                    alpha=0.3,
                    s=3
                )

                mean_value = mean_percentages.loc[freq]
                sem_value = sem_percentages.loc[freq]

                for sample_index, row in group_condition_data.iterrows():

                    condition_export_data.append({
                        'Condition': condition,
                        'Group': group,
                        'Sample_ID': row['Sample_ID'],
                        'Frequency_Hz': freq,
                        'Individual_Value': row[freq],
                        'Mean': mean_value,
                        'SEM': sem_value,
                        'Mean_minus_SEM': (
                            mean_value - sem_value
                            if pd.notna(sem_value)
                            else np.nan
                        ),
                        'Mean_plus_SEM': (
                            mean_value + sem_value
                            if pd.notna(sem_value)
                            else np.nan
                        )
                    })

        ax1.set_ylim(200, 400)
        ax2.set_ylim(-100, 200)

        ax2.set_xlim(0, 40)

        ax1.spines['bottom'].set_visible(False)
        ax2.spines['top'].set_visible(False)

        ax1.xaxis.tick_top()
        ax1.tick_params(labeltop=False)

        ax2.xaxis.tick_bottom()

        d = 0.01

        kwargs = dict(
            transform=ax1.transAxes,
            color='k',
            clip_on=False
        )

        ax1.plot(
            (-d, +d),
            (-d, +d),
            **kwargs
        )

        ax1.plot(
            (1 - d, 1 + d),
            (-d, +d),
            **kwargs
        )

        kwargs.update(
            transform=ax2.transAxes
        )

        ax2.plot(
            (-d, +d),
            (1 - d, 1 + d),
            **kwargs
        )

        ax2.plot(
            (1 - d, 1 + d),
            (1 - d, 1 + d),
            **kwargs
        )

        fig.suptitle(
            f'Averaged Normalized Frequency Domain Percentages '
            f'for Condition {condition}'
        )

        ax2.set_xlabel(
            'Frequency (Hz)'
        )

        ax1.set_ylabel(
            'Normalized Percentage (%)'
        )

        ax2.set_ylabel(
            'Normalized Percentage (%)'
        )

        ax2.legend(
            loc='upper left',
            bbox_to_anchor=(1, 1)
        )

        plt.tight_layout()
        plt.subplots_adjust(hspace=0.00)

        save_path = os.path.join(
            save_dir,
            f'Freq_Bin_condition_based_{condition}_Norm.pdf'
        )

        plt.savefig(
            save_path,
            format='pdf',
            bbox_inches='tight'
        )

        plt.close(fig)

        if len(condition_export_data) > 0:

            condition_export_df = pd.DataFrame(
                condition_export_data
            )

            condition_export_df = (
                condition_export_df
                .sort_values(
                    by=[
                        'Group',
                        'Frequency_Hz',
                        'Sample_ID'
                    ]
                )
                .reset_index(drop=True)
            )

            condition_export_df.to_excel(
                writer,
                sheet_name=condition[:31],
                index=False
            )

print()
print('Finished successfully.')
print()
print('PDF plots saved in:')
print(save_dir)
print()
print('Excel source data saved as:')
print(plot_data_excel)
print()
print('Each condition plot has its own Excel sheet.')
print('Only frequencies 0–40 Hz are included.')


## Selected Conditions ______ 05Plots_07_NormalizedFreqDomains Normalized Condition_based plots

- Uses the new excel file contain both normalized data and original data

- uses the already calculated normalized percentages of bins with specific freq in excel file


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

output_excel_path = NORMALIZED_FREQUENCY_XLSX
results_df = pd.read_excel(output_excel_path)

save_dir = RESULTS_DIR / "Selected_05Plots_05_FreqDomainsConditions"
os.makedirs(save_dir, exist_ok=True)

freq_data = []
for idx, row in results_df.iterrows():
    if row.get('File Name') != '20240812_18389_01':
        group = row['Group']
        condition = row['Condition']
        sample_id = int(row['Sample_ID'])

        freq_domain_dict = eval(row['Freq_Domain_Percentages_Norm'])
        freq_data.append({
            'Group': group, 'Condition': condition, 'Sample_ID': sample_id,
            **{int(k.strip('Hz')): float(v.strip('%')) for k, v in freq_domain_dict.items()}
        })

normalized_df = pd.DataFrame(freq_data)

normalized_df_filtered = normalized_df[normalized_df['Sample_ID'] != 18574].copy()

frequency_columns = normalized_df_filtered.columns[3:]

condition_order = ['Baseline', 'CBZ30uM', 'CBZ100uM']

normalized_df_filtered['Condition'] = pd.Categorical(normalized_df_filtered['Condition'], categories=condition_order, ordered=True)

def sem(x):
    return np.std(x, ddof=1) / np.sqrt(len(x))

unique_conditions = condition_order
groups = normalized_df_filtered['Group'].unique()

for condition in unique_conditions:
    fig, ax = plt.subplots(figsize=(4, 4))

    for group in groups:

        group_condition_data = normalized_df_filtered[
            (normalized_df_filtered['Group'] == group) &
            (normalized_df_filtered['Condition'] == condition)
        ]

        if not group_condition_data.empty:

            mean_percentages = group_condition_data[frequency_columns].mean()
            sem_percentages = group_condition_data[frequency_columns].apply(sem)
            freqs = np.arange(len(frequency_columns))

            plot_color = 'C1' if group == 'AAV-SMS' else 'C0'

            ax.plot(freqs, mean_percentages, label=f'{group} Mean', color=plot_color)
            ax.fill_between(freqs, mean_percentages - sem_percentages, mean_percentages + sem_percentages,
                            color=plot_color, alpha=0.2)

            for i, freq in enumerate(frequency_columns):
                freq_values = group_condition_data[freq].values
                ax.scatter([freqs[i]] * len(freq_values), freq_values, color=plot_color, alpha=0.3, s=3)

    ax.set_ylim(-120, 120)
    plt.xlim(0, 40)

    ax.set_title(f'Averaged Normalized Frequency Domain Percentages for Condition {condition}')
    ax.set_xlabel('Frequency (Hz)', fontsize=14)
    ax.set_ylabel('Normalized Percentage (%)', fontsize=14)

    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)

    ax.legend(loc='upper left', bbox_to_anchor=(1, 1))
    plt.tight_layout()

    save_path = os.path.join(save_dir, f'Freq_Bin_condition_based_{condition}_Norm.pdf')
    plt.savefig(save_path, format='pdf')
    plt.close(fig)


## 05_08_PlotsNet_FreqDomains Normalized Freq_Domain_Norm_5HzIntervals

- Uses NFreq_Domain_Percentage_Norm data, split them into 5Hz intervals (0 to 35Hz included)

- The new excel file containing the data of each Freq interval in each dpi is generated


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

output_excel_path = NORMALIZED_FREQUENCY_XLSX
results_df = pd.read_excel(output_excel_path)

save_dir = RESULTS_DIR / "05Plots_08_NormFreqDomainsConditions_intervals"
os.makedirs(save_dir, exist_ok=True)

freq_data = []
for idx, row in results_df.iterrows():
    if row.get('File Name') != '20240812_18389_01':
        group = row['Group']
        condition = row['Condition']
        sample_id = int(row['Sample_ID'])

        freq_domain_dict = eval(row['Freq_Domain_Percentages_Norm'])
        freq_data.append({
            'Group': group, 'Condition': condition, 'Sample_ID': sample_id,
            **{int(k.strip('Hz')): float(v.strip('%')) for k, v in freq_domain_dict.items()}
        })

normalized_df = pd.DataFrame(freq_data)

normalized_df_filtered = normalized_df[normalized_df['Sample_ID'] != 18574]

condition_order = ['Baseline', 'CBZ30uM', 'CBZ60uM', 'CBZ100uM', 'Washout']
normalized_df_filtered['Condition'] = pd.Categorical(
    normalized_df_filtered['Condition'], categories=condition_order, ordered=True
)

frequency_columns = normalized_df_filtered.columns[3:]
frequency_columns = [col for col in frequency_columns if isinstance(col, int) and col < 46]

bins = np.arange(0, 46, 5)

pooled_data = []
export_data_dict = {}

for i in range(len(bins) - 1):
    lower_bound = bins[i]
    upper_bound = bins[i + 1]

    interval_columns = [col for col in frequency_columns if lower_bound <= col < upper_bound]

    if interval_columns:
        for group in normalized_df_filtered['Group'].unique():
            for condition in condition_order:
                group_condition_data = normalized_df_filtered[
                    (normalized_df_filtered['Group'] == group) &
                    (normalized_df_filtered['Condition'] == condition)
                ]

                if not group_condition_data.empty:
                    frequency_data = group_condition_data[interval_columns]
                    mean_value = frequency_data.mean().mean()

                    pooled_data.append({
                        'Frequency Interval': f'{lower_bound}-{upper_bound}',
                        'Group': group,
                        'Condition': condition,
                        'Mean': mean_value,
                        'SD': frequency_data.std().mean(),
                        'SEM': frequency_data.std(ddof=1).mean() / np.sqrt(frequency_data.count().mean()),
                        'Data': frequency_data.values.flatten()
                    })

                    for _, sample_row in group_condition_data.iterrows():
                        sample_id = sample_row['Sample_ID']
                        key = (sample_id, condition, group)
                        if key not in export_data_dict:
                            export_data_dict[key] = {'Group': group, 'Sample_ID': sample_id, 'Condition': condition}
                        export_data_dict[key][f'{lower_bound}-{upper_bound} Hz'] = sample_row[interval_columns].mean()

pooled_df = pd.DataFrame(pooled_data)

export_df = pd.DataFrame.from_dict(export_data_dict, orient='index').reset_index(drop=True)

colors = {
    'AAV-Ctrl': '#1f77b4',
    'AAV-SMS': '#ff7f0e'
}

for condition in condition_order:
    fig, ax = plt.subplots(figsize=(10, 6))

    condition_data = pooled_df[pooled_df['Condition'] == condition]
    unique_intervals = condition_data['Frequency Interval'].unique()
    positions = np.arange(1, len(unique_intervals) + 1)

    for i, interval in enumerate(unique_intervals):
        interval_data = condition_data[condition_data['Frequency Interval'] == interval]
        box_data = []

        for group in interval_data['Group'].unique():
            data = interval_data[interval_data['Group'] == group]['Data'].values[0]
            box_data.append(data)

        box_positions = positions[i] + np.linspace(-0.2, 0.2, len(box_data))
        bp = ax.boxplot(box_data, positions=box_positions, widths=0.4, patch_artist=True)

        for patch, group in zip(bp['boxes'], interval_data['Group']):
            patch.set_facecolor(colors[group])
            patch.set_alpha(0.7)

        for median in bp['medians']:
            median.set_color('black')
            median.set_linewidth(1)

        for j, group in enumerate(interval_data['Group'].unique()):
            data_points = interval_data[interval_data['Group'] == group]['Data'].values[0]
            jittered_x = np.full(data_points.shape, box_positions[j]) + np.random.normal(0, 0.1, size=data_points.shape)
            jittered_y = data_points + np.random.normal(0, 2, size=data_points.shape)
            ax.scatter(jittered_x, jittered_y, alpha=0.6, color=colors[group], edgecolor='black', linewidth=0.5, s=10)

    ax.set_xticks(positions)
    ax.set_xticklabels(unique_intervals)
    ax.set_ylabel('Normalized Percentage (%)')
    ax.set_title(f'Boxplots of Frequency Domain Percentages by Interval - {condition}')
    ax.legend(colors.keys(), title="Group", loc='upper left', bbox_to_anchor=(1, 1))

    save_path = os.path.join(save_dir, f'Freq_Domain_{condition}_5HzIntervals.pdf')
    plt.tight_layout()
    plt.savefig(save_path, format='pdf')
    plt.show()
    plt.close(fig)

export_excel_path = os.path.join(save_dir, 'Freq_Domain_Norm_5HzIntervals.xlsx')
export_df.to_excel(export_excel_path, index=False)


## Selected Conditions ___ 05_08_PlotsNet_FreqDomains Normalized Freq_Domain_Norm_5HzIntervals

- Uses NFreq_Domain_Percentage_Norm data, split them into 5Hz intervals (0 to 35Hz included)

- The new excel file containing the data of each Freq interval in each dpi is generated


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

output_excel_path = NORMALIZED_FREQUENCY_XLSX
results_df = pd.read_excel(output_excel_path)

save_dir = RESULTS_DIR / "Selected_05Plots_05_FreqDomainsConditions"
os.makedirs(save_dir, exist_ok=True)

freq_data = []
for idx, row in results_df.iterrows():
    if row.get('File Name') != '20240812_18389_01':
        group = row['Group']
        condition = row['Condition']
        sample_id = int(row['Sample_ID'])

        freq_domain_dict = eval(row['Freq_Domain_Percentages_Norm'])
        freq_data.append({
            'Group': group, 'Condition': condition, 'Sample_ID': sample_id,
            **{int(k.strip('Hz')): float(v.strip('%')) for k, v in freq_domain_dict.items()}
        })

normalized_df = pd.DataFrame(freq_data)

normalized_df_filtered = normalized_df[normalized_df['Sample_ID'] != 18574]

condition_order = ['Baseline', 'CBZ30uM','CBZ100uM']
normalized_df_filtered['Condition'] = pd.Categorical(
    normalized_df_filtered['Condition'], categories=condition_order, ordered=True
)

frequency_columns = normalized_df_filtered.columns[3:]
frequency_columns = [col for col in frequency_columns if isinstance(col, int) and col < 41]

bins = np.arange(0, 41, 20)

pooled_data = []
export_data_dict = {}

for i in range(len(bins) - 1):
    lower_bound = bins[i]
    upper_bound = bins[i + 1]

    interval_columns = [col for col in frequency_columns if lower_bound <= col < upper_bound]

    if interval_columns:
        for group in normalized_df_filtered['Group'].unique():
            for condition in condition_order:
                group_condition_data = normalized_df_filtered[
                    (normalized_df_filtered['Group'] == group) &
                    (normalized_df_filtered['Condition'] == condition)
                ]

                if not group_condition_data.empty:
                    frequency_data = group_condition_data[interval_columns]
                    mean_value = frequency_data.mean().mean()

                    pooled_data.append({
                        'Frequency Interval': f'{lower_bound}-{upper_bound}',
                        'Group': group,
                        'Condition': condition,
                        'Mean': mean_value,
                        'SD': frequency_data.std().mean(),
                        'SEM': frequency_data.std(ddof=1).mean() / np.sqrt(frequency_data.count().mean()),
                        'Data': frequency_data.values.flatten()
                    })

                    for _, sample_row in group_condition_data.iterrows():
                        sample_id = sample_row['Sample_ID']
                        key = (sample_id, condition, group)
                        if key not in export_data_dict:
                            export_data_dict[key] = {'Group': group, 'Sample_ID': sample_id, 'Condition': condition}

                        for column in interval_columns:
                            export_data_dict[key][f'{column} Hz'] = sample_row[column]

pooled_df = pd.DataFrame(pooled_data)

export_df = pd.DataFrame.from_dict(export_data_dict, orient='index').reset_index(drop=True)

colors = {
    'AAV-Ctrl': '#1f77b4',
    'AAV-SMS': '#ff7f0e'
}

for condition in condition_order:
    fig, ax = plt.subplots(figsize=(4, 4))

    condition_data = pooled_df[pooled_df['Condition'] == condition]
    unique_intervals = condition_data['Frequency Interval'].unique()
    positions = np.arange(1, len(unique_intervals) + 1)

    for i, interval in enumerate(unique_intervals):
        interval_data = condition_data[condition_data['Frequency Interval'] == interval]
        box_data = []
        means = []
        sems = []

        for group in interval_data['Group'].unique():
            data = interval_data[interval_data['Group'] == group]['Data'].values[0]
            box_data.append(data)
            means.append(np.mean(data))
            sems.append(interval_data[interval_data['Group'] == group]['SEM'].values[0])

        box_positions = positions[i] + np.linspace(-0.2, 0.2, len(box_data))
        bp = ax.boxplot(box_data, positions=box_positions, widths=0.4, patch_artist=True)

        for patch, group in zip(bp['boxes'], interval_data['Group']):
            patch.set_facecolor(colors[group])
            patch.set_alpha(0.7)

        for median in bp['medians']:
            median.set_color('black')
            median.set_linewidth(1)

        for j, group in enumerate(interval_data['Group'].unique()):
            data_points = interval_data[interval_data['Group'] == group]['Data'].values[0]
            jittered_x = np.full(data_points.shape, box_positions[j]) + np.random.normal(0, 0.1, size=data_points.shape)
            jittered_y = data_points + np.random.normal(0, 2, size=data_points.shape)
            ax.scatter(jittered_x, jittered_y, alpha=0.6, color=colors[group], edgecolor='black', linewidth=0.5, s=10)

        for j, (mean_value, sem_value) in enumerate(zip(means, sems)):
            ax.errorbar(box_positions[j], mean_value, yerr=sem_value, fmt='o',
                        color=colors[interval_data['Group'].unique()[j]],
                        ecolor='black', elinewidth=1, capsize=3, markersize=8,
                        markeredgewidth=1.5, markeredgecolor='black')

    ax.set_xticks(positions)
    ax.set_xticklabels(unique_intervals)
    plt.ylim(-120, 120)

    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)
    ax.set_ylabel('Normalized Percentage (%)', fontsize=14)
    ax.set_title(f'Boxplots of Frequency Domain Percentages by Interval - {condition}')
    ax.legend(colors.keys(), title="Group", loc='upper left', bbox_to_anchor=(1, 1))

    save_path = os.path.join(save_dir, f'Freq_Domain_{condition}_20HzIntervals.pdf')
    plt.tight_layout()
    plt.savefig(save_path, format='pdf')
    plt.show()
    plt.close(fig)

export_excel_path = os.path.join(save_dir, 'Freq_Domain_Norm_IndividualData_20HzIntervals.xlsx')
export_df.to_excel(export_excel_path, index=False)


## 05_09_PlotsNet_FreqDomains Normalized Freq_Domain_Norm_5HzIntervals ---- Outliers Removed

- Uses NFreq_Domain_Percentage_Norm data, split them into 5Hz intervals (0 to 35Hz included)

- The new excel file containing the data of each Freq interval in each dpi is generated


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

output_excel_path = NORMALIZED_FREQUENCY_XLSX
results_df = pd.read_excel(output_excel_path)

save_dir = RESULTS_DIR / "05Plots_09_NormFreqDomainsConditions_intervals_outliersremoved"
os.makedirs(save_dir, exist_ok=True)

freq_data = []
for idx, row in results_df.iterrows():
    if row.get('File Name') != '20240812_18389_01':
        group = row['Group']
        condition = row['Condition']
        sample_id = int(row['Sample_ID'])

        freq_domain_dict = eval(row['Freq_Domain_Percentages_Norm'])
        freq_data.append({
            'Group': group, 'Condition': condition, 'Sample_ID': sample_id,
            **{int(k.strip('Hz')): float(v.strip('%')) for k, v in freq_domain_dict.items()}
        })

normalized_df = pd.DataFrame(freq_data)

normalized_df_filtered = normalized_df[normalized_df['Sample_ID'] != 18574]

condition_order = ['Baseline', 'CBZ30uM', 'CBZ60uM', 'CBZ100uM', 'Washout']
normalized_df_filtered['Condition'] = pd.Categorical(
    normalized_df_filtered['Condition'], categories=condition_order, ordered=True
)

frequency_columns = normalized_df_filtered.columns[3:]
frequency_columns = [col for col in frequency_columns if isinstance(col, int) and col < 46]

bins = np.arange(0, 46, 5)

pooled_data = []
export_data_dict = {}

for i in range(len(bins) - 1):
    lower_bound = bins[i]
    upper_bound = bins[i + 1]

    interval_columns = [col for col in frequency_columns if lower_bound <= col < upper_bound]

    if interval_columns:
        for group in normalized_df_filtered['Group'].unique():
            for condition in condition_order:
                group_condition_data = normalized_df_filtered[
                    (normalized_df_filtered['Group'] == group) &
                    (normalized_df_filtered['Condition'] == condition)
                ]

                if not group_condition_data.empty:

                    all_data_points = group_condition_data[interval_columns].values.flatten()

                    Q1 = np.percentile(all_data_points, 25)
                    Q3 = np.percentile(all_data_points, 75)
                    IQR = Q3 - Q1
                    non_outlier_data = all_data_points[(all_data_points >= Q1 - 1.5 * IQR) & (all_data_points <= Q3 + 1.5 * IQR)]

                    mean_value = np.mean(non_outlier_data)
                    sd_value = np.std(non_outlier_data)
                    sem_value = sd_value / np.sqrt(len(non_outlier_data))

                    pooled_data.append({
                        'Frequency Interval': f'{lower_bound}-{upper_bound}',
                        'Group': group,
                        'Condition': condition,
                        'Mean': mean_value,
                        'SD': sd_value,
                        'SEM': sem_value,
                        'Data': non_outlier_data
                    })

                    for _, sample_row in group_condition_data.iterrows():
                        sample_id = sample_row['Sample_ID']
                        key = (sample_id, condition, group)
                        if key not in export_data_dict:
                            export_data_dict[key] = {'Group': group, 'Sample_ID': sample_id, 'Condition': condition}
                        export_data_dict[key][f'{lower_bound}-{upper_bound} Hz'] = sample_row[interval_columns].mean()

pooled_df = pd.DataFrame(pooled_data)

export_df = pd.DataFrame.from_dict(export_data_dict, orient='index').reset_index(drop=True)

colors = {
    'AAV-Ctrl': '#1f77b4',
    'AAV-SMS': '#ff7f0e'
}

for condition in condition_order:
    fig, ax = plt.subplots(figsize=(10, 6))

    condition_data = pooled_df[pooled_df['Condition'] == condition]
    unique_intervals = condition_data['Frequency Interval'].unique()
    positions = np.arange(1, len(unique_intervals) + 1)

    for i, interval in enumerate(unique_intervals):
        interval_data = condition_data[condition_data['Frequency Interval'] == interval]
        box_data = []

        for group in interval_data['Group'].unique():
            data = interval_data[interval_data['Group'] == group]['Data'].values[0]
            box_data.append(data)

        box_positions = positions[i] + np.linspace(-0.2, 0.2, len(box_data))
        bp = ax.boxplot(box_data, positions=box_positions, widths=0.4, patch_artist=True)

        for patch, group in zip(bp['boxes'], interval_data['Group']):
            patch.set_facecolor(colors[group])
            patch.set_alpha(0.7)

        for median in bp['medians']:
            median.set_color('black')
            median.set_linewidth(1)

        for j, group in enumerate(interval_data['Group'].unique()):
            data_points = interval_data[interval_data['Group'] == group]['Data'].values[0]
            jittered_x = np.full(data_points.shape, box_positions[j]) + np.random.normal(0, 0.1, size=data_points.shape)
            jittered_y = data_points + np.random.normal(0, 2, size=data_points.shape)
            ax.scatter(jittered_x, jittered_y, alpha=0.6, color=colors[group], edgecolor='black', linewidth=0.5, s=10)

    ax.set_xticks(positions)
    ax.set_xticklabels(unique_intervals)
    ax.set_ylabel('Normalized Percentage (%)')
    ax.set_title(f'Boxplots of Frequency Domain Percentages by Interval - {condition}')
    ax.legend(colors.keys(), title="Group", loc='upper left', bbox_to_anchor=(1, 1))

    save_path = os.path.join(save_dir, f'Freq_Domain_{condition}_5HzIntervals.pdf')
    plt.tight_layout()
    plt.savefig(save_path, format='pdf')
    plt.show()
    plt.close(fig)

export_excel_path = os.path.join(save_dir, 'Freq_Domain_Norm_5HzIntervals_outliersremoved.xlsx')
export_df.to_excel(export_excel_path, index=False)


## Connectivity Analysis


## Tau shift calculation


In [ ]:
import pandas as pd
import numpy as np
from scipy.signal import correlate
import os
import pickle
import zipfile
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

pickle_file_path = BASELINE_CONNECTIVITY_PKL
output_excel_path = RESULTS_DIR / "tau_matrices.xlsx"
output_pdf_dir = RESULTS_DIR / "Tau_Plots"
os.makedirs(output_pdf_dir, exist_ok=True)

writer = pd.ExcelWriter(output_excel_path, engine='openpyxl')

sampling_frequency = 1

def calculate_tau_matrix(timeseries_data, neuron_ids, sampling_frequency):
    num_neurons = len(neuron_ids)
    tau_matrix = np.zeros((num_neurons, num_neurons))

    spike_trains = {neuron: np.array(times) / sampling_frequency for neuron, times in timeseries_data.items()}

    for i in range(num_neurons):
        for j in range(i + 1, num_neurons):
            spike_times_i = spike_trains[neuron_ids[i]]
            spike_times_j = spike_trains[neuron_ids[j]]

            bin_size = 0.01
            max_time = max(spike_times_i.max(), spike_times_j.max())
            bin_edges = np.arange(0, max_time + bin_size, bin_size)

            binned_train_i, _ = np.histogram(spike_times_i, bins=bin_edges)
            binned_train_j, _ = np.histogram(spike_times_j, bins=bin_edges)

            cross_corr = correlate(binned_train_i - np.mean(binned_train_i),
                                   binned_train_j - np.mean(binned_train_j), mode='full')
            lags = np.arange(-len(binned_train_i) + 1, len(binned_train_i))
            tau = lags[np.argmax(np.abs(cross_corr))] * bin_size

            tau_matrix[i, j] = tau
            tau_matrix[j, i] = -tau

    return tau_matrix

def process_sample(data, sample_id, dpi, writer):

    sample_data = data[(data['Sample_ID'] == sample_id) & (data['dpi'] == dpi)]
    neuron_ids = sample_data['unit_id'].astype(str).tolist()
    timeseries_data = {str(row['unit_id']): row['Timeseries'] for _, row in sample_data.iterrows()}

    tau_matrix = calculate_tau_matrix(timeseries_data, neuron_ids, sampling_frequency)

    sheet_name = f'Sample_{sample_id}_dpi_{dpi}'
    tau_df = pd.DataFrame(tau_matrix, index=neuron_ids, columns=neuron_ids)
    tau_df.to_excel(writer, sheet_name=sheet_name)

    pdf_path = os.path.join(output_pdf_dir, f'{sheet_name}.pdf')
    with PdfPages(pdf_path) as pdf:
        plt.figure(figsize=(14, 6))

        plt.subplot(1, 2, 1)
        plt.imshow(tau_matrix, cmap='coolwarm', interpolation='none', aspect='auto')
        plt.colorbar(label='Time Shift (Tau) in seconds')
        plt.xticks(ticks=np.arange(len(neuron_ids)), labels=neuron_ids, rotation=90)
        plt.yticks(ticks=np.arange(len(neuron_ids)), labels=neuron_ids)
        plt.title(f'Time Shift (Tau) - {sheet_name}')

        threshold_matrix = np.where((tau_matrix >= -0.01) & (tau_matrix <= 0.01), 1, np.nan)

        plt.subplot(1, 2, 2)
        plt.imshow(threshold_matrix, cmap='gray', interpolation='none', aspect='auto')
        plt.colorbar(label='Tau within [-10ms, 10ms]')
        plt.xticks(ticks=np.arange(len(neuron_ids)), labels=neuron_ids, rotation=90)
        plt.yticks(ticks=np.arange(len(neuron_ids)), labels=neuron_ids)
        plt.title(f'Thresholded Tau - {sheet_name}')

        plt.tight_layout()
        pdf.savefig()
        plt.close()

with zipfile.ZipFile(pickle_file_path, 'r') as zip_file:
    with zip_file.open(zip_file.namelist()[0]) as file:
        data = pickle.load(file)

data = pd.DataFrame(data)

for (sample_id, dpi) in data[['Sample_ID', 'dpi']].drop_duplicates().itertuples(index=False):
    process_sample(data, sample_id, dpi, writer)

writer.close()
print(f"Tau matrices and plots have been saved.")


## Tau shift calculation with matrix export for many samples in pickle file


In [ ]:
import os
import gzip
import pickle
import pandas as pd
import numpy as np
from scipy.signal import correlate
from statsmodels.tsa.vector_ar.var_model import VAR

def bin_timeseries(timeseries, bin_size=0.05, total_time=600):
    """
    Consider that the timeseries have been or have not been modifed based on sampling frequency
    timeseries: list of spike times for a single neuron.
    bin_size: bin width in ms (default 50 ms).
    total_time: total duration in s (600 seconds).
    """
    num_bins = int(total_time // bin_size)
    binned_series = np.zeros(num_bins)
    for spike_time in timeseries:
        bin_idx = int(spike_time // bin_size)
        if bin_idx < num_bins:
            binned_series[bin_idx] += 1
    return binned_series

def smooth_data(data, window_size=5):
    """
    Smooth the data using a simple moving average.
    data: binned spike counts.
    window_size: the size of the moving average window.
    """
    return np.convolve(data, np.ones(window_size)/window_size, mode='same')

def calculate_cross_correlation_matrix(binned_data):
    """
    Calculate the cross-correlation matrix for binned spike data.
    """
    num_neurons = len(binned_data)
    cc_matrix = np.zeros((num_neurons, num_neurons))

    for i in range(num_neurons):
        for j in range(num_neurons):
            if i != j:
                cc = np.corrcoef(binned_data[i], binned_data[j])[0, 1]
                cc_matrix[i, j] = cc
            else:
                cc_matrix[i, j] = 1

    return cc_matrix

def calculate_granger_causality_matrix(binned_data, maxlag=20):
    """
    Calculate the Granger causality matrix for binned spike data.
    MAXLAG: is the max number of bins before that are used for prediction here 20 x 50ms bin = 1000ms before is used to interpret upcoming timepoint
    Lags:
      When assessing Granger causality, you are not only looking at the present values of the time series but also their past values (lags).
      For example, if you set maxlag=10, the model will consider how past values of both time series up to 10 time steps ago influence the current
      value of the series being predicted.
    Model Complexity:
      A higher value for maxlag means that the model will use more past values, which can improve prediction accuracy if there is
      significant information in those lags. However, it also increases the complexity of the model and the amount of data required to fit the model
      reliably.
    Trade-offs:
    - If maxlag is too high, the model might overfit, especially if the time series is short or if there are not enough data points to
      reliably estimate the parameters of the model.
    - If maxlag is too low, you might miss out on important lags that influence the relationship.
    """
    num_neurons = len(binned_data)
    gc_matrix = np.zeros((num_neurons, num_neurons))

    for i in range(num_neurons):
        for j in range(num_neurons):
            if i != j:
                combined_data = np.vstack([binned_data[i], binned_data[j]]).T
                model = VAR(combined_data)
                results = model.fit(maxlags=maxlag)
                gc_matrix[i, j] = results.test_causality(0, 1, kind='f').test_statistic

    return gc_matrix

def process_and_save_sample(sample_data, group, sample_id, dpi, output_folder, dt, total_time=600):
    """
    Process a single sample, calculate CC and GC matrices, and save as .csv files.
    """

    metadata_rows = sample_data[['File Name', 'post_SMS', 'Date', 'x', 'y', 'unit_id']].copy()
    metadata_rows['Group'] = group
    metadata_rows['Sample_ID'] = sample_id
    metadata_rows['dpi'] = dpi

    timeseries_data = {row['unit_id']: row['Timeseries'] for _, row in sample_data.iterrows()}
    binned_data = [bin_timeseries(timeseries_data[neuron], bin_size=dt, total_time=total_time)
                   for neuron in timeseries_data]

    smoothed_binned_data = [smooth_data(data) for data in binned_data]

    cc_matrix = calculate_cross_correlation_matrix(smoothed_binned_data)

    gc_matrix = calculate_granger_causality_matrix(smoothed_binned_data, maxlag=20)

    cc_df = pd.DataFrame(cc_matrix, index=timeseries_data.keys(), columns=timeseries_data.keys())
    gc_df = pd.DataFrame(gc_matrix, index=timeseries_data.keys(), columns=timeseries_data.keys())

    cc_result_df = metadata_rows.set_index('unit_id').join(cc_df, how='right').reset_index()
    cc_output_filename = os.path.join(output_folder, f"{group}_{sample_id}_{dpi}_CC.csv")
    cc_result_df.to_csv(cc_output_filename, index=False)
    print(f"Saved Cross-Correlation results to {cc_output_filename}")

    gc_result_df = metadata_rows.set_index('unit_id').join(gc_df, how='right').reset_index()
    gc_output_filename = os.path.join(output_folder, f"{group}_{sample_id}_{dpi}_GC.csv")
    gc_result_df.to_csv(gc_output_filename, index=False)
    print(f"Saved Granger Causality results to {gc_output_filename}")

def main(input_pickle_path, output_folder, dt=0.05):

    with gzip.open(input_pickle_path, 'rb') as f:
        data = pickle.load(f)

    print("Data loaded, shape:", data.shape)

    for group in data['Group'].unique():
        group_data = data[data['Group'] == group]

        for sample_id in group_data['Sample_ID'].unique():
            sample_data = group_data[group_data['Sample_ID'] == sample_id]

            for dpi in sample_data['dpi'].unique():
                dpi_data = sample_data[sample_data['dpi'] == dpi]

                process_and_save_sample(dpi_data, group, sample_id, dpi, output_folder, dt)

input_pickle_path = BASELINE_CCGC_PKL
output_folder = RESULTS_DIR / "CCGC_Baseline_SMS"
os.makedirs(output_folder, exist_ok=True)

main(input_pickle_path, output_folder)


## 06_01_Crosscorrelation (+/- correlations) & Granger casuality (directionality )

CBZ

Applys parallelization to spedd up the process

With Neuron number selection options (random selection)

With sample selection options


In [ ]:
import os
import gzip
import pickle
import pandas as pd
import numpy as np
from scipy.signal import correlate
from joblib import Parallel, delayed
from statsmodels.tsa.vector_ar.var_model import VAR
from scipy.stats import pearsonr

def bin_timeseries(timeseries, bin_size=0.05, total_time=600):
    num_bins = int(total_time // bin_size)
    binned_series = np.zeros(num_bins)
    for spike_time in timeseries:
        bin_idx = int(spike_time // bin_size)
        if bin_idx < num_bins:
            binned_series[bin_idx] += 1
    return binned_series

def smooth_data(data, window_size=5):
    return np.convolve(data, np.ones(window_size) / window_size, mode='same')

def calculate_cross_correlation_row(i, binned_data):
    num_neurons = len(binned_data)
    cc_row = np.zeros(num_neurons)
    for j in range(num_neurons):
        if i != j:
            cc_row[j] = pearsonr(binned_data[i], binned_data[j])[0]
        else:
            cc_row[j] = 1
    return cc_row

def calculate_cross_correlation_matrix(binned_data):
    num_neurons = len(binned_data)
    cc_matrix = Parallel(n_jobs=-1)(delayed(calculate_cross_correlation_row)(i, binned_data)
                                    for i in range(num_neurons))
    return np.array(cc_matrix)

def calculate_granger_causality_row(i, binned_data, maxlag=20):
    num_neurons = len(binned_data)
    gc_stat_row = np.zeros(num_neurons)
    gc_pval_row = np.zeros(num_neurons)
    for j in range(num_neurons):
        if i != j:
            combined_data = np.vstack([binned_data[i], binned_data[j]]).T
            model = VAR(combined_data)
            results = model.fit(maxlags=maxlag)
            test_result = results.test_causality(0, 1, kind='f')
            gc_stat_row[j] = test_result.test_statistic
            gc_pval_row[j] = test_result.pvalue
    return gc_stat_row, gc_pval_row

def calculate_granger_causality_matrix(binned_data, maxlag=20):
    num_neurons = len(binned_data)
    results = Parallel(n_jobs=-1)(delayed(calculate_granger_causality_row)(i, binned_data, maxlag)
                                  for i in range(num_neurons))
    gc_stat_matrix, gc_pval_matrix = zip(*results)
    return np.array(gc_stat_matrix), np.array(gc_pval_matrix)

def process_and_save_sample(sample_data, group, sample_id, condition, output_folder, dt, total_time=600, max_neurons=500):
    metadata_rows = sample_data[['File Name', 'post_SMS', 'Date', 'x', 'y', 'unit_id']].copy()
    metadata_rows['Group'] = group
    metadata_rows['Sample_ID'] = sample_id
    metadata_rows['Condition'] = condition

    timeseries_data = {row['unit_id']: row['Timeseries'] for _, row in sample_data.iterrows()}
    if len(timeseries_data) > max_neurons:
        selected_neurons = np.random.choice(list(timeseries_data.keys()), max_neurons, replace=False)
        timeseries_data = {neuron: timeseries_data[neuron] for neuron in selected_neurons}

    binned_data = [bin_timeseries(timeseries_data[neuron], bin_size=dt, total_time=total_time)
                   for neuron in timeseries_data]
    smoothed_binned_data = [smooth_data(data) for data in binned_data]

    cc_matrix = calculate_cross_correlation_matrix(smoothed_binned_data)
    gc_matrix, gc_pval_matrix = calculate_granger_causality_matrix(smoothed_binned_data, maxlag=20)

    cc_df = pd.DataFrame(cc_matrix, index=timeseries_data.keys(), columns=timeseries_data.keys())
    gc_df = pd.DataFrame(gc_matrix, index=timeseries_data.keys(), columns=timeseries_data.keys())
    gc_pval_df = pd.DataFrame(gc_pval_matrix, index=timeseries_data.keys(), columns=timeseries_data.keys())

    cc_result_df = metadata_rows.set_index('unit_id').join(cc_df, how='right').reset_index()
    cc_output_filename = os.path.join(output_folder, f"{group}_{sample_id}_{condition}_CC.csv")
    cc_result_df.to_csv(cc_output_filename, index=False)

    gc_result_df = metadata_rows.set_index('unit_id').join(gc_df, how='right').reset_index()
    gc_output_filename = os.path.join(output_folder, f"{group}_{sample_id}_{condition}_GC.csv")
    gc_result_df.to_csv(gc_output_filename, index=False)

    gc_pval_result_df = metadata_rows.set_index('unit_id').join(gc_pval_df, how='right').reset_index()
    gc_pval_output_filename = os.path.join(output_folder, f"{group}_{sample_id}_{condition}_GC_pvalues.csv")
    gc_pval_result_df.to_csv(gc_pval_output_filename, index=False)

    print(f"Saved Cross-Correlation, Granger Causality, and p-value results for {group}, sample {sample_id}, condition {condition}")

exclude_criteria = [
    {'Sample_ID': '18389', 'Condition': ['Baseline', 'CBZ30uM', 'CBZ60uM', 'CBZ100uM', 'Washout']},

    {'Sample_ID': '18505', 'Condition': ['Baseline', 'CBZ30uM', 'CBZ60uM', 'CBZ100uM', 'Washout']},
    {'Sample_ID': '20810', 'Condition': ['Baseline', 'CBZ30uM', 'CBZ60uM', 'CBZ100uM', 'Washout']},
    {'Sample_ID': '18574', 'Condition': ['Baseline', 'CBZ30uM', 'CBZ60uM', 'CBZ100uM', 'Washout']},
]

def should_exclude(sample_id, condition, exclude_criteria):
    for criterion in exclude_criteria:
        if criterion['Sample_ID'] == sample_id and condition in criterion['Condition']:
            print(sample_id, condition, 'excluded')
            return True
    return False

def main(input_pickle_path, output_folder, dt=0.05):
    with gzip.open(input_pickle_path, 'rb') as f:
        data = pickle.load(f)

    print("Data loaded, shape:", data.shape)

    for group in data['Group'].unique():
        group_data = data[data['Group'] == group]

        for sample_id in group_data['Sample_ID'].unique():
            sample_data = group_data[group_data['Sample_ID'] == sample_id]

            for condition in sample_data['Condition'].unique():
                condition_data = sample_data[sample_data['Condition'] == condition]

                if should_exclude(sample_id, condition, exclude_criteria):
                    print(f"Skipping Sample_ID {sample_id} with condition {condition} due to exclusion criteria.")
                    continue

                process_and_save_sample(condition_data, group, sample_id, condition, output_folder, dt)

input_pickle_path = UNIT_TIMESERIES_PKL
output_folder = CONNECTIVITY_MATRIX_DIR
os.makedirs(output_folder, exist_ok=True)

main(input_pickle_path, output_folder)


## 07_01_Measure of Connectivity parameters 01


## Measurement of Fast connectivity metrices (metrices that can be calcuated faster) in network

Neuron_Based connectivity measures

For processing many files

Data of all Neurons, in all Samples and Condition are collected in one .CSV file in one sheet

Later the .csv output of this code will be used for MEA_Based analysis or normalized data


In [ ]:
import os
import pandas as pd
import numpy as np
import networkx as nx
import community as community_louvain

input_directory = CONNECTIVITY_MATRIX_DIR
output_file = CONNECTIVITY_PARAMETERS_CSV
csv_files = [f for f in os.listdir(input_directory) if f.endswith('.csv')]

all_results = []

for file_name in csv_files:

    file_path = os.path.join(input_directory, file_name)
    data = pd.read_csv(file_path)

    metadata_columns = data.columns[:data.columns.get_loc("Condition") + 1]
    metadata = data[metadata_columns]
    correlation_matrix = data.drop(columns=metadata_columns)

    np.fill_diagonal(correlation_matrix.values, 0)
    correlation_matrix = pd.DataFrame(np.where(correlation_matrix <= 0.3, 0, correlation_matrix),
                                  columns=correlation_matrix.columns,
                                  index=correlation_matrix.index)

    G = nx.from_numpy_array(correlation_matrix.values)
    for u, v, data in G.edges(data=True):
        data['weight'] = correlation_matrix.iloc[u, v]

    if G.number_of_edges() > 0:

        partition = community_louvain.best_partition(G, weight='weight', resolution=1)
        modularity_value = community_louvain.modularity(partition, G)
    else:

        partition = {}
        modularity_value = None

    def calculate_neuron_params(G):
        num_nodes = len(G.nodes())
        degree = dict(G.degree(weight=None))
        density = {node: deg / (num_nodes - 1) for node, deg in degree.items()}
        weight_sum = {node: sum(abs(data['weight']) for _, _, data in G.edges(node, data=True)) for node in G.nodes()}
        weight_mean = {node: weight_sum[node] / (num_nodes - 1) for node in G.nodes()}
        clustering_coefficients = nx.clustering(G, weight='weight')
        eigenvector_centrality = nx.eigenvector_centrality(G, weight='weight') if G.number_of_edges() > 0 else {}
        page_rank = nx.pagerank(G, weight='weight') if G.number_of_edges() > 0 else {}
        avg_neighbor_degree = nx.average_neighbor_degree(G, weight='weight') if G.number_of_edges() > 0 else {}
        assortativity = nx.degree_assortativity_coefficient(G, weight='weight') if G.number_of_edges() > 0 else None
        avg_path_length = {}
        for node in G.nodes():
            path_lengths = nx.single_source_dijkstra_path_length(G, node, weight='weight')
            avg_path_length[node] = np.mean(list(path_lengths.values()))
        strength = {node: sum(data['weight'] for _, _, data in G.edges(node, data=True)) for node in G.nodes()}
        closeness_centrality = nx.closeness_centrality(G, distance='weight') if G.number_of_edges() > 0 else {}
        degree_mean = np.mean(list(degree.values()))
        degree_std = np.std(list(degree.values()))
        z_score_degree = {node: (deg - degree_mean) / degree_std for node, deg in degree.items()}
        transitivity = nx.transitivity(G) if G.number_of_edges() > 0 else None

        return {
            'Degree': degree,
            'Density': density,
            'Weight_sum': weight_sum,
            'Weight_mean': weight_mean,
            'Clustering_coefficient': clustering_coefficients,
            'Eigenvector_centrality': eigenvector_centrality,
            'PageRank': page_rank,
            'Average_neighbor_degree': avg_neighbor_degree,
            'Assortativity': assortativity,
            'Average_path_length': avg_path_length,
            'Strength': strength,
            'Closeness_centrality': closeness_centrality,
            'Z_Score_of_Degree': z_score_degree,
            'Transitivity': {node: transitivity for node in G.nodes()},
            'Community': partition,
            'Modularity': {node: modularity_value for node in G.nodes()}
        }

    neuron_params = calculate_neuron_params(G)

    neuron_params_df = pd.DataFrame(neuron_params)
    neuron_params_df.index.name = 'Neuron'
    neuron_params_df.reset_index(inplace=True)
    final_df = pd.concat([metadata.reset_index(drop=True), neuron_params_df], axis=1)
    final_df['Source_File'] = file_name

    all_results.append(final_df)

combined_df = pd.concat(all_results, ignore_index=True)
combined_df.to_csv(output_file, index=False)

print(f"All results successfully saved to {output_file}")


## 07Plots_01_MEABasedConnectivityParams_01 condition ---- CBZ 01 ---- Plot & ---ExcelExport---

The code has been replicated using "04Plots_01_MEABasedBurst_01 condition — CBZ 01 — Plot & — ExcelExport — " code


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from scipy.stats import skew, kurtosis, mode

csv_file = CONNECTIVITY_PARAMETERS_CSV
final_df = pd.read_csv(csv_file)

final_df = final_df[final_df['Sample_ID'] != 18574]

output_dir = RESULTS_DIR / "07Plots_01_MEABasedConnectivityParams_01"
os.makedirs(output_dir, exist_ok=True)

variables = [
    'Degree', 'Density', 'Weight_sum', 'Weight_mean', 'Clustering_coefficient',
    'Eigenvector_centrality', 'PageRank', 'Average_neighbor_degree', 'Assortativity',
    'Average_path_length', 'Strength', 'Closeness_centrality', 'Z_Score_of_Degree',
    'Transitivity', 'Modularity'
]

condition_order = ['Baseline', 'CBZ30uM', 'CBZ60uM', 'CBZ100uM', 'Washout']

filtered_df = final_df

sample_stats_per_condition = {}

import warnings

for sample_id in filtered_df['Sample_ID'].unique():
    sample_df = filtered_df[filtered_df['Sample_ID'] == sample_id]
    sample_condition_stats = {}

    for condition in sample_df['Condition'].unique():
        condition_df = sample_df[sample_df['Condition'] == condition]

        condition_stats = {}
        for var in variables:
            values = condition_df[var].dropna()

            if values.empty:
                mean_val = sdv_val = sem_val = median_val = max_val = min_val = cv_val = 0
                variance_val = percentile_25 = percentile_50 = percentile_75 = 0
                skewness_val = kurtosis_val = mode_val = avg_z_score = 0
            else:

                mean_val = values.mean()
                sdv_val = values.std()
                sem_val = sdv_val / np.sqrt(len(values))
                median_val = values.median()
                max_val = values.max()
                min_val = values.min()
                cv_val = sdv_val / mean_val if mean_val != 0 else np.nan
                variance_val = np.var(values)
                percentile_25 = np.percentile(values, 25)
                percentile_50 = np.percentile(values, 50)
                percentile_75 = np.percentile(values, 75)

                with warnings.catch_warnings():
                    warnings.simplefilter("ignore", category=RuntimeWarning)

                    if variance_val > 1e-10:
                        skewness_val = skew(values)
                        kurtosis_val = kurtosis(values)
                    else:
                        skewness_val = kurtosis_val = 0

                mode_val = mode(values)[0][0] if isinstance(mode(values)[0], np.ndarray) else mode(values)[0]
                z_scores = (values - mean_val) / sdv_val
                avg_z_score = np.mean(z_scores)

            condition_stats[var] = {
                'mean': mean_val,
                'sdv': sdv_val,
                'sem': sem_val,
                'median': median_val,
                'max': max_val,
                'min': min_val,
                'cv': cv_val,
                'variance': variance_val,
                'percentile_25': percentile_25,
                'percentile_50': percentile_50,
                'percentile_75': percentile_75,
                'skewness': skewness_val,
                'kurtosis': kurtosis_val,
                'mode': mode_val,
                'average_z_score': avg_z_score
            }

        sample_condition_stats[condition] = condition_stats

    sample_stats_per_condition[sample_id] = sample_condition_stats

group_stats = {}
for group in filtered_df['Group'].unique():
    group_df = filtered_df[filtered_df['Group'] == group]

    group_avg_stats = {}
    for condition in group_df['Condition'].unique():
        samples_in_group_condition = [
            sample_stats_per_condition[sample_id][condition]
            for sample_id in group_df[group_df['Condition'] == condition]['Sample_ID'].unique()
        ]
        condition_group_stats = {}

        for var in variables:

            sample_means = [sample[var]['mean'] for sample in samples_in_group_condition]
            mean_val = np.mean(sample_means)
            sem_val = np.std(sample_means) / np.sqrt(len(sample_means))

            condition_group_stats[var] = {
                'mean': mean_val,
                'sem': sem_val,
                'sample_means': sample_means
            }

        group_avg_stats[condition] = condition_group_stats

    group_stats[group] = group_avg_stats

group_colors = {
    'AAV-Ctrl': '#1f77b4',
    'AAV-SMS': '#ff7f0e'
}

for var in variables:
    plt.figure(figsize=(10, 6))

    for group_idx, (group, group_data) in enumerate(group_stats.items()):
        conditions = [condition for condition in condition_order if condition in group_data]
        color = group_colors.get(group, 'black')

        boxplot_data = [group_data[condition][var]['sample_means'] for condition in conditions]

        x_positions = np.arange(len(conditions)) + group_idx * 0.2
        box = plt.boxplot(
            boxplot_data,
            positions=x_positions,
            widths=0.3,
            patch_artist=True,
            boxprops=dict(facecolor=color, edgecolor='black', linewidth=0.5),
            medianprops=dict(color='black', linewidth=1),
            whiskerprops=dict(color='black', linewidth=0.2),
            capprops=dict(color='black', linewidth=0.2),
            flierprops=dict(marker='o', color=color, markersize=5, alpha=0.5),
            zorder=1
        )

        means = [group_data[condition][var]['mean'] for condition in conditions]
        plt.plot(x_positions, means, marker='o', color=color,
                 label=f'{group} Mean', linewidth=1, linestyle='-',
                 markersize=6, markeredgecolor='black', zorder=2)

        for pos, sample_means in zip(x_positions, boxplot_data):
            jittered_x = pos + np.random.normal(0, 0.05, size=len(sample_means))
            plt.scatter(jittered_x, sample_means, color=color, edgecolor='black',
                        alpha=0.7, linewidth=0.2, zorder=3)

    plt.title(f'{var} (Box Plot with Scatter Points and Mean Lines) across Conditions')
    plt.xlabel('Condition')
    plt.ylabel(var)

    plt.legend(title="Groups", bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.xticks(ticks=np.arange(len(conditions)), labels=conditions)

    plot_output_path = os.path.join(output_dir, f'{var}_group_comparison_boxplot.pdf')
    plt.savefig(plot_output_path, format='pdf', bbox_inches='tight')
    plt.close()

output_excel_file = RESULTS_DIR / "07_01_MEABasedConnectivityParamsStat_01.xlsx"
with pd.ExcelWriter(output_excel_file) as writer:
    for var in variables:
        var_stats = []
        for sample_id, sample_stats in sample_stats_per_condition.items():
            for condition, condition_stats in sample_stats.items():
                metadata_row = filtered_df[filtered_df['Sample_ID'] == sample_id].iloc[0]

                var_stats.append({
                    'Group': metadata_row['Group'],
                    'Sample_ID': sample_id,
                    'Date': metadata_row['Date'],
                    'Condition': condition,
                    'mean': condition_stats[var]['mean'],
                    'sdv': condition_stats[var]['sdv'],
                    'sem': condition_stats[var]['sem'],
                    'median': condition_stats[var]['median'],
                    'max': condition_stats[var]['max'],
                    'min': condition_stats[var]['min'],
                    'cv': condition_stats[var]['cv'],
                    'variance': condition_stats[var]['variance'],
                    'percentile_25': condition_stats[var]['percentile_25'],
                    'percentile_50': condition_stats[var]['percentile_50'],
                    'percentile_75': condition_stats[var]['percentile_75'],
                    'skewness': condition_stats[var]['skewness'],
                    'kurtosis': condition_stats[var]['kurtosis'],
                    'mode': condition_stats[var]['mode'],
                    'average_z_score': condition_stats[var]['average_z_score']
                })

        var_df = pd.DataFrame(var_stats)
        var_df.to_excel(writer, sheet_name=var[:31], index=False)

print("Code execution completed.")


## 07Plots_01_MEABasedConnectivityParams_01 condition ---- CBZ 01 ---- Plot & ---ExcelExport---

The code has been replicated using "04Plots_01_MEABasedBurst_01 condition — CBZ 01 — Plot & — ExcelExport — " code


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from scipy.stats import skew, kurtosis, mode

csv_file = CONNECTIVITY_PARAMETERS_CSV
final_df = pd.read_csv(csv_file)

final_df = final_df[final_df['Sample_ID'] != 18574]

output_dir = RESULTS_DIR / "Selected_07Plots_01_MEABasedConnectivityParams_01"
os.makedirs(output_dir, exist_ok=True)

variables = [
    'Degree', 'Density', 'Weight_sum', 'Weight_mean', 'Clustering_coefficient',
    'Eigenvector_centrality', 'PageRank', 'Average_neighbor_degree', 'Assortativity',
    'Average_path_length', 'Strength', 'Closeness_centrality', 'Z_Score_of_Degree',
    'Transitivity', 'Modularity'
]

condition_order = ['Baseline', 'CBZ30uM','CBZ100uM']

filtered_df = final_df

sample_stats_per_condition = {}

import warnings

for sample_id in filtered_df['Sample_ID'].unique():
    sample_df = filtered_df[filtered_df['Sample_ID'] == sample_id]
    sample_condition_stats = {}

    for condition in sample_df['Condition'].unique():
        condition_df = sample_df[sample_df['Condition'] == condition]

        condition_stats = {}
        for var in variables:
            values = condition_df[var].dropna()

            if values.empty:
                mean_val = sdv_val = sem_val = median_val = max_val = min_val = cv_val = 0
                variance_val = percentile_25 = percentile_50 = percentile_75 = 0
                skewness_val = kurtosis_val = mode_val = avg_z_score = 0
            else:

                mean_val = values.mean()
                sdv_val = values.std()
                sem_val = sdv_val / np.sqrt(len(values))
                median_val = values.median()
                max_val = values.max()
                min_val = values.min()
                cv_val = sdv_val / mean_val if mean_val != 0 else np.nan
                variance_val = np.var(values)
                percentile_25 = np.percentile(values, 25)
                percentile_50 = np.percentile(values, 50)
                percentile_75 = np.percentile(values, 75)

                with warnings.catch_warnings():
                    warnings.simplefilter("ignore", category=RuntimeWarning)

                    if variance_val > 1e-10:
                        skewness_val = skew(values)
                        kurtosis_val = kurtosis(values)
                    else:
                        skewness_val = kurtosis_val = 0

                mode_val = mode(values)[0][0] if isinstance(mode(values)[0], np.ndarray) else mode(values)[0]
                z_scores = (values - mean_val) / sdv_val
                avg_z_score = np.mean(z_scores)

            condition_stats[var] = {
                'mean': mean_val,
                'sdv': sdv_val,
                'sem': sem_val,
                'median': median_val,
                'max': max_val,
                'min': min_val,
                'cv': cv_val,
                'variance': variance_val,
                'percentile_25': percentile_25,
                'percentile_50': percentile_50,
                'percentile_75': percentile_75,
                'skewness': skewness_val,
                'kurtosis': kurtosis_val,
                'mode': mode_val,
                'average_z_score': avg_z_score
            }

        sample_condition_stats[condition] = condition_stats

    sample_stats_per_condition[sample_id] = sample_condition_stats

group_stats = {}
for group in filtered_df['Group'].unique():
    group_df = filtered_df[filtered_df['Group'] == group]

    group_avg_stats = {}
    for condition in group_df['Condition'].unique():
        samples_in_group_condition = [
            sample_stats_per_condition[sample_id][condition]
            for sample_id in group_df[group_df['Condition'] == condition]['Sample_ID'].unique()
        ]
        condition_group_stats = {}

        for var in variables:

            sample_means = [sample[var]['mean'] for sample in samples_in_group_condition]
            mean_val = np.mean(sample_means)
            sem_val = np.std(sample_means) / np.sqrt(len(sample_means))

            condition_group_stats[var] = {
                'mean': mean_val,
                'sem': sem_val,
                'sample_means': sample_means
            }

        group_avg_stats[condition] = condition_group_stats

    group_stats[group] = group_avg_stats

group_colors = {
    'AAV-Ctrl': '#1f77b4',
    'AAV-SMS': '#ff7f0e'
}

for var in variables:
    plt.figure(figsize=(4, 6))

    for group_idx, (group, group_data) in enumerate(group_stats.items()):
        conditions = [condition for condition in condition_order if condition in group_data]
        color = group_colors.get(group, 'black')

        boxplot_data = [group_data[condition][var]['sample_means'] for condition in conditions]

        x_positions = np.arange(len(conditions)) + group_idx * 0.25
        box = plt.boxplot(
            boxplot_data,
            positions=x_positions,
            widths=0.25,
            patch_artist=True,
            boxprops=dict(facecolor=color, edgecolor='black', linewidth=0.7, alpha=0.7),
            medianprops=dict(color='black', linewidth=1),
            whiskerprops=dict(color='black', linewidth=0.7),
            capprops=dict(color='black', linewidth=0.7),
            flierprops=dict(marker='o', color=color, markersize=5, alpha=0.5),
            zorder=1
        )

        means = [group_data[condition][var]['mean'] for condition in conditions]
        plt.plot(x_positions, means, marker='o', color=color,
                 label=f'{group} Mean', linewidth=1.5, linestyle='-',
                 markersize=8, markeredgecolor='black', zorder=2)

        for pos, sample_means in zip(x_positions, boxplot_data):
            jittered_x = pos + np.random.normal(0, 0.05, size=len(sample_means))
            plt.scatter(jittered_x, sample_means, color=color, edgecolor='black',
                        alpha=0.7, linewidth=0.2, zorder=3, s=40)

    plt.title(f'{var} (Box Plot with Scatter Points and Mean Lines) across Conditions')
    plt.xlabel('Condition', fontsize=14)
    plt.ylabel(var)

    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)

    plt.legend(title="Groups", bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=14)
    plt.xticks(ticks=np.arange(len(conditions)), labels=conditions, fontsize=14)

    plot_output_path = os.path.join(output_dir, f'{var}_Boxplot.pdf')
    plt.savefig(plot_output_path, format='pdf', bbox_inches='tight')
    plt.close()

output_excel_file = RESULTS_DIR / "Selected_07_01_MEABasedConnectivityParamsStat_01.xlsx"
with pd.ExcelWriter(output_excel_file) as writer:
    for var in variables:
        var_stats = []
        for sample_id, sample_stats in sample_stats_per_condition.items():
            for condition, condition_stats in sample_stats.items():
                metadata_row = filtered_df[filtered_df['Sample_ID'] == sample_id].iloc[0]

                var_stats.append({
                    'Group': metadata_row['Group'],
                    'Sample_ID': sample_id,
                    'Date': metadata_row['Date'],
                    'Condition': condition,
                    'mean': condition_stats[var]['mean'],
                    'sdv': condition_stats[var]['sdv'],
                    'sem': condition_stats[var]['sem'],
                    'median': condition_stats[var]['median'],
                    'max': condition_stats[var]['max'],
                    'min': condition_stats[var]['min'],
                    'cv': condition_stats[var]['cv'],
                    'variance': condition_stats[var]['variance'],
                    'percentile_25': condition_stats[var]['percentile_25'],
                    'percentile_50': condition_stats[var]['percentile_50'],
                    'percentile_75': condition_stats[var]['percentile_75'],
                    'skewness': condition_stats[var]['skewness'],
                    'kurtosis': condition_stats[var]['kurtosis'],
                    'mode': condition_stats[var]['mode'],
                    'average_z_score': condition_stats[var]['average_z_score']
                })

        var_df = pd.DataFrame(var_stats)
        var_df.to_excel(writer, sheet_name=var[:31], index=False)

print("Code execution completed.")


## 07Plots_02_Normalized MEABasedConnectivityParams_01 condition ---- CBZ 01 ---- Plot & ---ExcelExport---


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from scipy.stats import skew, kurtosis, mode

csv_file = CONNECTIVITY_PARAMETERS_CSV
final_df = pd.read_csv(csv_file)

final_df = final_df[final_df['Sample_ID'] != 18574]

output_dir = RESULTS_DIR / "07Plots_02_NormMEABasedConnectivityParams_01"
os.makedirs(output_dir, exist_ok=True)

variables = [
    'Degree', 'Density', 'Weight_sum', 'Weight_mean', 'Clustering_coefficient',
    'Eigenvector_centrality', 'PageRank', 'Average_neighbor_degree', 'Assortativity',
    'Average_path_length', 'Strength', 'Closeness_centrality', 'Z_Score_of_Degree',
    'Transitivity', 'Modularity'
]

condition_order = ['Baseline', 'CBZ30uM', 'CBZ60uM', 'CBZ100uM', 'Washout']

filtered_df = final_df

sample_stats_per_condition = {}

for sample_id in filtered_df['Sample_ID'].unique():
    sample_df = filtered_df[filtered_df['Sample_ID'] == sample_id]

    baseline_means = {}
    baseline_df = sample_df[sample_df['Condition'] == 'Baseline']
    for var in variables:
        baseline_means[var] = baseline_df[var].mean() if not baseline_df[var].isna().all() else np.nan

    sample_condition_stats = {}

    for condition in sample_df['Condition'].unique():
        condition_df = sample_df[sample_df['Condition'] == condition]

        condition_stats = {}
        for var in variables:
            values = condition_df[var].dropna()

            baseline_mean = baseline_means[var]
            if not np.isnan(baseline_mean) and baseline_mean != 0:
                normalized_values = values / baseline_mean
            else:
                normalized_values = values

            if normalized_values.empty:
                mean_val = sdv_val = sem_val = median_val = max_val = min_val = cv_val = 0
                variance_val = percentile_25 = percentile_50 = percentile_75 = 0
                skewness_val = kurtosis_val = mode_val = avg_z_score = 0
            else:

                mean_val = normalized_values.mean()
                sdv_val = normalized_values.std()
                sem_val = sdv_val / np.sqrt(len(normalized_values))
                median_val = normalized_values.median()
                max_val = normalized_values.max()
                min_val = normalized_values.min()
                cv_val = sdv_val / mean_val if mean_val != 0 else np.nan
                variance_val = np.var(normalized_values)
                percentile_25 = np.percentile(normalized_values, 25)
                percentile_50 = np.percentile(normalized_values, 50)
                percentile_75 = np.percentile(normalized_values, 75)

                with warnings.catch_warnings():
                    warnings.simplefilter("ignore", category=RuntimeWarning)
                    if variance_val > 1e-10:
                        skewness_val = skew(normalized_values)
                        kurtosis_val = kurtosis(normalized_values)
                    else:
                        skewness_val = kurtosis_val = 0

                mode_val = mode(normalized_values)[0][0] if isinstance(mode(normalized_values)[0], np.ndarray) else mode(normalized_values)[0]
                z_scores = (normalized_values - mean_val) / sdv_val
                avg_z_score = np.mean(z_scores)

            condition_stats[var] = {
                'mean': mean_val,
                'sdv': sdv_val,
                'sem': sem_val,
                'median': median_val,
                'max': max_val,
                'min': min_val,
                'cv': cv_val,
                'variance': variance_val,
                'percentile_25': percentile_25,
                'percentile_50': percentile_50,
                'percentile_75': percentile_75,
                'skewness': skewness_val,
                'kurtosis': kurtosis_val,
                'mode': mode_val,
                'average_z_score': avg_z_score
            }

        sample_condition_stats[condition] = condition_stats

    sample_stats_per_condition[sample_id] = sample_condition_stats

group_stats = {}
for group in filtered_df['Group'].unique():
    group_df = filtered_df[filtered_df['Group'] == group]

    group_avg_stats = {}
    for condition in group_df['Condition'].unique():
        samples_in_group_condition = [
            sample_stats_per_condition[sample_id][condition]
            for sample_id in group_df[group_df['Condition'] == condition]['Sample_ID'].unique()
        ]
        condition_group_stats = {}

        for var in variables:

            sample_means = [sample[var]['mean'] for sample in samples_in_group_condition]
            mean_val = np.mean(sample_means)
            sem_val = np.std(sample_means) / np.sqrt(len(sample_means))

            condition_group_stats[var] = {
                'mean': mean_val,
                'sem': sem_val,
                'sample_means': sample_means
            }

        group_avg_stats[condition] = condition_group_stats

    group_stats[group] = group_avg_stats

group_colors = {
    'AAV-Ctrl': '#1f77b4',
    'AAV-SMS': '#ff7f0e'
}

for var in variables:
    plt.figure(figsize=(10, 6))

    for group_idx, (group, group_data) in enumerate(group_stats.items()):
        conditions = [condition for condition in condition_order if condition in group_data]
        color = group_colors.get(group, 'black')

        boxplot_data = [group_data[condition][var]['sample_means'] for condition in conditions]

        x_positions = np.arange(len(conditions)) + group_idx * 0.2
        box = plt.boxplot(
            boxplot_data,
            positions=x_positions,
            widths=0.3,
            patch_artist=True,
            boxprops=dict(facecolor=color, edgecolor='black', linewidth=0.5),
            medianprops=dict(color='black', linewidth=1),
            whiskerprops=dict(color='black', linewidth=0.2),
            capprops=dict(color='black', linewidth=0.2),
            flierprops=dict(marker='o', color=color, markersize=5, alpha=0.5),
            zorder=1
        )

        means = [group_data[condition][var]['mean'] for condition in conditions]
        plt.plot(x_positions, means, marker='o', color=color,
                 label=f'{group} Mean', linewidth=1, linestyle='-',
                 markersize=6, markeredgecolor='black', zorder=2)

        for pos, sample_means in zip(x_positions, boxplot_data):
            jittered_x = pos + np.random.normal(0, 0.05, size=len(sample_means))
            plt.scatter(jittered_x, sample_means, color=color, edgecolor='black',
                        alpha=0.7, linewidth=0.2, zorder=3)

    plt.title(f'{var} (Box Plot with Scatter Points and Mean Lines) across Conditions')
    plt.xlabel('Condition')
    plt.ylabel(var)

    plt.legend(title="Groups", bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.xticks(ticks=np.arange(len(conditions)), labels=conditions)

    plot_output_path = os.path.join(output_dir, f'{var}_group_comparison_boxplot.pdf')
    plt.savefig(plot_output_path, format='pdf', bbox_inches='tight')
    plt.close()

output_excel_file = RESULTS_DIR / "07_02_NormalizedMEABasedConnectivityParamsStat_01.xlsx"
with pd.ExcelWriter(output_excel_file) as writer:
    for var in variables:
        var_stats = []
        for sample_id, sample_stats in sample_stats_per_condition.items():
            for condition, condition_stats in sample_stats.items():
                metadata_row = filtered_df[filtered_df['Sample_ID'] == sample_id].iloc[0]

                var_stats.append({
                    'Group': metadata_row['Group'],
                    'Sample_ID': sample_id,
                    'Date': metadata_row['Date'],
                    'Condition': condition,
                    'mean': condition_stats[var]['mean'],
                    'sdv': condition_stats[var]['sdv'],
                    'sem': condition_stats[var]['sem'],
                    'median': condition_stats[var]['median'],
                    'max': condition_stats[var]['max'],
                    'min': condition_stats[var]['min'],
                    'cv': condition_stats[var]['cv'],
                    'variance': condition_stats[var]['variance'],
                    'percentile_25': condition_stats[var]['percentile_25'],
                    'percentile_50': condition_stats[var]['percentile_50'],
                    'percentile_75': condition_stats[var]['percentile_75'],
                    'skewness': condition_stats[var]['skewness'],
                    'kurtosis': condition_stats[var]['kurtosis'],
                    'mode': condition_stats[var]['mode'],
                    'average_z_score': condition_stats[var]['average_z_score']
                })

        var_df = pd.DataFrame(var_stats)
        var_df.to_excel(writer, sheet_name=var[:31], index=False)

print("Code execution completed.")


## Selected Conditions____ Normalized___07Plots_02_Normalized MEABasedConnectivityParams_01

07Plots_02_Normalized MEABasedConnectivityParams_01 condition — CBZ 01 — Plot & — ExcelExport — 


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from scipy.stats import skew, kurtosis, mode
import warnings

csv_file = CONNECTIVITY_PARAMETERS_CSV
final_df = pd.read_csv(csv_file)

final_df = final_df[final_df['Sample_ID'] != 18574]

output_dir = RESULTS_DIR / "Selected_07Plots_01_MEABasedConnectivityParams_01"
os.makedirs(output_dir, exist_ok=True)

variables = [
    'Degree', 'Density', 'Weight_sum', 'Weight_mean', 'Clustering_coefficient',
    'Eigenvector_centrality', 'PageRank', 'Average_neighbor_degree', 'Assortativity',
    'Average_path_length', 'Strength', 'Closeness_centrality', 'Z_Score_of_Degree',
    'Transitivity', 'Modularity'
]

condition_order = ['Baseline', 'CBZ30uM', 'CBZ100uM']

filtered_df = final_df

sample_stats_per_condition = {}

for sample_id in filtered_df['Sample_ID'].unique():
    sample_df = filtered_df[filtered_df['Sample_ID'] == sample_id]

    baseline_means = {}
    baseline_df = sample_df[sample_df['Condition'] == 'Baseline']
    for var in variables:
        baseline_means[var] = baseline_df[var].mean() if not baseline_df[var].isna().all() else np.nan

    sample_condition_stats = {}

    for condition in sample_df['Condition'].unique():
        condition_df = sample_df[sample_df['Condition'] == condition]

        condition_stats = {}
        for var in variables:
            values = condition_df[var].dropna()

            baseline_mean = baseline_means[var]
            if not np.isnan(baseline_mean) and baseline_mean != 0:
                normalized_values = ((values - baseline_mean) / baseline_mean) * 100
            else:
                normalized_values = values

            if normalized_values.empty:
                mean_val = sdv_val = sem_val = median_val = max_val = min_val = cv_val = 0
                variance_val = percentile_25 = percentile_50 = percentile_75 = 0
                skewness_val = kurtosis_val = mode_val = avg_z_score = 0
            else:

                mean_val = normalized_values.mean()
                sdv_val = normalized_values.std()
                sem_val = sdv_val / np.sqrt(len(normalized_values))
                median_val = normalized_values.median()
                max_val = normalized_values.max()
                min_val = normalized_values.min()
                cv_val = sdv_val / mean_val if mean_val != 0 else np.nan
                variance_val = np.var(normalized_values)
                percentile_25 = np.percentile(normalized_values, 25)
                percentile_50 = np.percentile(normalized_values, 50)
                percentile_75 = np.percentile(normalized_values, 75)

                with warnings.catch_warnings():
                    warnings.simplefilter("ignore", category=RuntimeWarning)
                    if variance_val > 1e-10:
                        skewness_val = skew(normalized_values)
                        kurtosis_val = kurtosis(normalized_values)
                    else:
                        skewness_val = kurtosis_val = 0

                mode_val = mode(normalized_values)[0][0] if isinstance(mode(normalized_values)[0], np.ndarray) else mode(normalized_values)[0]
                z_scores = (normalized_values - mean_val) / sdv_val
                avg_z_score = np.mean(z_scores)

            condition_stats[var] = {
                'mean': mean_val,
                'sdv': sdv_val,
                'sem': sem_val,
                'median': median_val,
                'max': max_val,
                'min': min_val,
                'cv': cv_val,
                'variance': variance_val,
                'percentile_25': percentile_25,
                'percentile_50': percentile_50,
                'percentile_75': percentile_75,
                'skewness': skewness_val,
                'kurtosis': kurtosis_val,
                'mode': mode_val,
                'average_z_score': avg_z_score
            }

        sample_condition_stats[condition] = condition_stats

    sample_stats_per_condition[sample_id] = sample_condition_stats

group_stats = {}
for group in filtered_df['Group'].unique():
    group_df = filtered_df[filtered_df['Group'] == group]

    group_avg_stats = {}
    for condition in group_df['Condition'].unique():
        samples_in_group_condition = [
            sample_stats_per_condition[sample_id][condition]
            for sample_id in group_df[group_df['Condition'] == condition]['Sample_ID'].unique()
        ]
        condition_group_stats = {}

        for var in variables:

            sample_means = [sample[var]['mean'] for sample in samples_in_group_condition]
            mean_val = np.mean(sample_means)
            sem_val = np.std(sample_means) / np.sqrt(len(sample_means))

            condition_group_stats[var] = {
                'mean': mean_val,
                'sem': sem_val,
                'sample_means': sample_means
            }

        group_avg_stats[condition] = condition_group_stats

    group_stats[group] = group_avg_stats

group_colors = {
    'AAV-Ctrl': '#1f77b4',
    'AAV-SMS': '#ff7f0e'
}

for var in variables:
    plt.figure(figsize=(4, 4))

    for group_idx, (group, group_data) in enumerate(group_stats.items()):
        conditions = [condition for condition in condition_order if condition in group_data]
        color = group_colors.get(group, 'black')

        boxplot_data = [group_data[condition][var]['sample_means'] for condition in conditions]

        x_positions = np.arange(len(conditions)) + group_idx * 0.25
        box = plt.boxplot(
            boxplot_data,
            positions=x_positions,
            widths=0.25,
            patch_artist=True,
            boxprops=dict(facecolor=color, edgecolor='black', linewidth=0.7, alpha=0.7),
            medianprops=dict(color='black', linewidth=1),
            whiskerprops=dict(color='black', linewidth=0.7),
            capprops=dict(color='black', linewidth=0.7),
            flierprops=dict(marker='o', color=color, markersize=5, alpha=0.5),
            zorder=1
        )

        means = [group_data[condition][var]['mean'] for condition in conditions]
        plt.plot(x_positions, means, marker='o', color=color,
                 label=f'{group} Mean', linewidth=1.5, linestyle='-',
                 markersize=8, markeredgecolor='black', zorder=2)

        for pos, sample_means in zip(x_positions, boxplot_data):
            jittered_x = pos + np.random.normal(0, 0.05, size=len(sample_means))
            plt.scatter(jittered_x, sample_means, color=color, edgecolor='black',
                        alpha=0.7, linewidth=0.2, zorder=3, s=40)

    plt.title(f'{var} (Box Plot with Scatter Points and Mean Lines) across Conditions')
    plt.xlabel('Condition', fontsize=14)
    plt.ylabel(var)
    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)

    plt.legend(title="Groups", bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=14)
    plt.xticks(ticks=np.arange(len(conditions)), labels=conditions, fontsize=14)

    plot_output_path = os.path.join(output_dir, f'{var}_NormalizedBoxplot.pdf')
    plt.savefig(plot_output_path, format='pdf', bbox_inches='tight')
    plt.close()

output_excel_file = RESULTS_DIR / "Selected_07_02_NormalizedMEABasedConnectivityParamsStat_01.xlsx"
with pd.ExcelWriter(output_excel_file) as writer:
    for var in variables:
        var_stats = []
        for sample_id, sample_stats in sample_stats_per_condition.items():
            for condition, condition_stats in sample_stats.items():
                metadata_row = filtered_df[filtered_df['Sample_ID'] == sample_id].iloc[0]

                var_stats.append({
                    'Group': metadata_row['Group'],
                    'Sample_ID': sample_id,
                    'Date': metadata_row['Date'],
                    'Condition': condition,
                    'mean': condition_stats[var]['mean'],
                    'sdv': condition_stats[var]['sdv'],
                    'sem': condition_stats[var]['sem'],
                    'median': condition_stats[var]['median'],
                    'max': condition_stats[var]['max'],
                    'min': condition_stats[var]['min'],
                    'cv': condition_stats[var]['cv'],
                    'variance': condition_stats[var]['variance'],
                    'percentile_25': condition_stats[var]['percentile_25'],
                    'percentile_50': condition_stats[var]['percentile_50'],
                    'percentile_75': condition_stats[var]['percentile_75'],
                    'skewness': condition_stats[var]['skewness'],
                    'kurtosis': condition_stats[var]['kurtosis'],
                    'mode': condition_stats[var]['mode'],
                    'average_z_score': condition_stats[var]['average_z_score']
                })

        var_df = pd.DataFrame(var_stats)
        var_df.to_excel(writer, sheet_name=var[:31], index=False)

print("Code execution completed.")


## 08Plots_01_ConnectivityMaps_01 condition ---- CBZ 01

Maps the connectivity and export PDF PNG and Video MP4 Files

Negative Nodes and Negative edges can be plotted in different color

Threshold allows to set simultaneous positive and negative thresholds

Nodes with no edge were also plotted to visualize density of connections


In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import os
from pathlib import Path
import cv2

x_min, x_max = -100, 4000
y_min, y_max = -100, 2250

condition_order = ['Baseline', 'CBZ30uM', 'CBZ60uM', 'CBZ100uM', 'Washout']

csv_directory = CONNECTIVITY_MATRIX_DIR
output_directory = RESULTS_DIR / "08Plots_01_Connectivity_Maps_Geometric_01"
output_directory.mkdir(parents=True, exist_ok=True)

def process_csv(file_path):
    data = pd.read_csv(file_path)

    group = data['Group'].iloc[0]
    sample_id = data['Sample_ID'].iloc[0]
    condition = data['Condition'].iloc[0]

    metadata_columns = data.columns[:data.columns.get_loc("Condition") + 1]

    correlation_matrix = data.drop(columns=metadata_columns)

    correlation_matrix.index = data['index'].values
    correlation_matrix.columns = data['index'].values

    x_positions = data['x'].values
    y_positions = data['y'].values
    neuron_names = data['index'].values

    np.fill_diagonal(correlation_matrix.values, 0)

    correlation_matrix = correlation_matrix.applymap(lambda x: 0 if -0.1 <= x <= 0.2 else x)

    G = nx.Graph()
    for i, (x, y) in enumerate(zip(x_positions, y_positions)):
        neuron_name = neuron_names[i]
        G.add_node(neuron_name, pos=(x, y))

    for i, row_neuron in enumerate(correlation_matrix.index):
        for j, col_neuron in enumerate(correlation_matrix.columns):
            weight = correlation_matrix.loc[row_neuron, col_neuron]
            if weight != 0:
                G.add_edge(row_neuron, col_neuron, weight=weight)

    min_size, max_size = 40, 100
    degrees = [G.degree(node) for node in G.nodes()]
    min_degree, max_degree = min(degrees), max(degrees)
    node_sizes = [
        min_size if deg == 0 else min_size + (max_size - min_size) * (deg - min_degree) / (max_degree - min_degree)
        for deg in degrees
    ]

    node_colors = []
    for node in G.nodes():
        edges = G.edges(node, data=True)
        num_edges = len(edges)
        num_negative_edges = sum(1 for _, _, data in edges if data['weight'] < 0)
        negative_percentage = (num_negative_edges / num_edges) * 100 if num_edges > 0 else 0
        node_colors.append('orange' if negative_percentage > 99 else 'orange')

    fig, ax = plt.subplots(figsize=(19.20, 10.80))
    fig.patch.set_facecolor('black')
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_facecolor('black')

    pos = {node: (x, y) for node, x, y in zip(neuron_names, x_positions, y_positions)}
    draw_edges(ax, G, pos)

    for idx, node in enumerate(G.nodes()):
        x, y = pos[node]
        ax.scatter(x, y, s=node_sizes[idx], color=node_colors[idx], edgecolors='black', alpha=0.8, zorder=2)

    ax.set_title(f"Neuronal Connectivity Map - {condition}", color='white')
    ax.set_axis_off()

    return fig, ax, group, sample_id, condition

def draw_edges(ax, G, pos):
    for u, v, data in G.edges(data=True):
        if u != v:
            x1, y1 = pos[u]
            x2, y2 = pos[v]
            if data['weight'] < 2:
                edge_color = 'gray'
                edge_alpha = min(1, abs(data['weight']) * 0.1)
                edge_width = max(0.5, abs(data['weight'] * 0.2))
            else:
                edge_color = 'gray'
                edge_alpha = 0.2
                edge_width = abs(data['weight'] * 0.2)
            ax.plot([x1, x2], [y1, y2], color=edge_color, alpha=edge_alpha, linewidth=edge_width, zorder=1)

def add_transition_frames(start_img, end_img, num_transition_frames=10):
    transition_frames = []
    for i in range(1, num_transition_frames + 1):
        alpha = i / (num_transition_frames + 1)
        transition_frame = cv2.addWeighted(start_img, 1 - alpha, end_img, alpha, 0)
        transition_frames.append(transition_frame)
    return transition_frames

def generate_plots_and_videos():
    samples = {}

    for file_path in csv_directory.glob("*.csv"):
        fig, ax, group, sample_id, condition = process_csv(file_path)

        filename = f"{group}_{sample_id}_{condition}"

        fig.savefig(output_directory / f"{filename}.png", transparent=False)
        fig.savefig(output_directory / f"{filename}.pdf", transparent=False)

        sample_key = f"{group}_{sample_id}"
        if sample_key not in samples:
            samples[sample_key] = []
        samples[sample_key].append((condition, fig))

        plt.close(fig)

    for sample_key, conditions in samples.items():

        sorted_conditions = sorted(conditions, key=lambda x: condition_order.index(x[0]))

        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        video_path = output_directory / f"{sample_key}_connectivity_video.mp4"
        video_writer = cv2.VideoWriter(str(video_path), fourcc, 1, (1920, 1080))

        previous_img = None
        for condition, fig in sorted_conditions:

            temp_png = output_directory / f"{sample_key}_{condition}.png"
            fig.savefig(temp_png, transparent=False)

            img = cv2.imread(str(temp_png))
            img_resized = cv2.resize(img, (1920, 1080))

            if previous_img is not None:
                transition_frames = add_transition_frames(previous_img, img_resized)
                for frame in transition_frames:
                    video_writer.write(frame)

            video_writer.write(img_resized)
            previous_img = img_resized

        video_writer.release()

generate_plots_and_videos()


## Reproducibility and interpretation

This notebook is intended to reproduce the analysis workflow used for the manuscript rather than to serve as a general-purpose software package. Experimental group definitions, condition labels, sample identifiers, exclusion rules, and analysis thresholds are therefore retained as explicit parts of the workflow. Users applying the code to a different dataset should review those experiment-specific settings before execution.

The analysis should be run sequentially because several later sections use intermediate files produced by earlier sections. All generated outputs are written beneath `results/`, while source data remain under `data/`.
